# 14. Canonical Pipeline Aggregate — Herbal Supplements

This notebook is the single aggregation authority for the Herbal Supplements retrieve-then-rerank experiment. It combines the two frozen Notebook 09 candidate interfaces with the Shannon Entropy, LightGBM, GAM, and Transformer outputs and creates one canonical per-case rank and metric layer.

The canonical design contains six machine conditions: P0, P1-only, P2-Q, P2-P, b02, and Full. Their report labels are S1-Q, S1-P, Base, RankP, RetrP, and Full. The training-free Shannon family has no separate RetrP fit, so the Shannon–b02 cell is excluded by design. The resulting primary method grid therefore contains 23 condition–family cells rather than a complete Cartesian product.

All rank-based metrics are reconstructed from the authoritative one-based target rank. An unretrieved target receives zero. The headline Stage 2 endpoint is unconditional NDCG@5 at candidate depth 1,000; depths 100, 300, 500, and 700 remain fixed-prefix sensitivity interfaces. The stored execution retains 1,968 cases and exports the canonical raw table, the complete primary grid, aggregate summaries, source inventories, disagreement diagnostics, hashes, and the downstream manifest.

The stored run recorded 114 upstream metric-field disagreements and retained the rank-derived reconstruction as canonical. These records document source-field inconsistency; they do not alter the canonical ranks or the reconstructed metrics.

The learned Full conditions are fitted on the personalized candidate interface under the current own-pool design. However, several executable provenance fields retain earlier transport-oriented names, including `full_transfer_readiness` and `zero_training_transfer_verified`. In addition, the LightGBM and Transformer Full model-contract hash paths still resolve through the RankP/query-only contract location. These legacy fields must not be interpreted as evidence that Full reused a query-only model or as complete proof of Full model-artifact lineage.

The received notebook contains 14 cells: one Markdown cell and 13 executed code cells. Nine code cells contain stored outputs, no stored error is present, and execution counts are sequential from 1 through 13.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ==== Imports ====
from pathlib import Path
from datetime import datetime, timezone
import gc
import itertools
import hashlib
import json
import math
import os
import re
import shutil
import uuid

import numpy as np
import pandas as pd


In [3]:
# ==== Canonical Experiment Contract and Exact Upstream Paths ====
NOTEBOOK_NAME = "14_pipeline_aggregate_herbal.ipynb"
CATEGORY_ID = "herbal"
CATEGORY_KEY = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"

PROJECT_ROOT = Path(f"/content/drive/MyDrive/thesis_recsys/categories/{CATEGORY_KEY}")
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
ANALYSIS_DIR = OUTPUTS_DIR / "analysis"
OUT_DIR = OUTPUTS_DIR / "pipeline_aggregate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = uuid.uuid4().hex
CANONICAL_MANIFEST_PATH = OUT_DIR / "pipeline_manifest.json"
RUN_STAGING_DIR = OUT_DIR / f".batch3_staging_{RUN_ID}"
PREVIOUS_MANIFEST_BACKUP_PATH = OUT_DIR / "pipeline_manifest.previous_success.json"
RUN_STAGING_DIR.mkdir(parents=True, exist_ok=False)

if CANONICAL_MANIFEST_PATH.exists():
    try:
        _previous_manifest = json.loads(
            CANONICAL_MANIFEST_PATH.read_text(encoding="utf-8")
        )
    except Exception:
        _previous_manifest = None
    if isinstance(_previous_manifest, dict) and _previous_manifest.get("run_status") == "SUCCESS":
        shutil.copy2(CANONICAL_MANIFEST_PATH, PREVIOUS_MANIFEST_BACKUP_PATH)

CANONICAL_MANIFEST_PATH.write_text(
    json.dumps(
        {
            "run_status": "IN_PROGRESS",
            "ready_for_downstream": False,
            "run_id": RUN_ID,
            "notebook_name": NOTEBOOK_NAME,
            "category_id": CATEGORY_ID,
            "publication_state": "canonical_outputs_not_promoted",
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

STAGE1_MANIFEST_PATH = (
    OUTPUTS_DIR
    / "stage1_candidate_pools"
    / f"stage1_candidate_pool_export_manifest_{CATEGORY_ID}.json"
)

CANONICAL_CONDITIONS = ["P0", "P1-only", "P2-Q", "P2-P", "b02", "Full"]
RERANKER_FAMILIES = ["shannon", "lightgbm", "gam", "transformer"]
CANDIDATE_POOL_DEPTHS = [100, 300, 500, 700, 1000]
METRIC_CUTOFFS = [1, 5, 10, 100, 300, 500, 700, 1000]
METRIC_NAMES = ["HitRate", "NDCG", "MRR"]
GAM_AVAILABLE_CANDIDATE_POOL_DEPTHS = list(CANDIDATE_POOL_DEPTHS)
SOURCE_METRIC_TOLERANCE = 1e-8
REPORT_POOL_DEPTH = 1000  # report depth 1000 (2026-07-23); reranking always top-1000; all-depth panels unchanged
FROZEN_FALLBACK_CASE_COUNT = 1415
PRIMARY_METRIC_NAME = "NDCG"
PRIMARY_METRIC_CUTOFF = 5
FULL_VALIDATION_REQUIRED_FAMILIES = {"lightgbm", "transformer"}
EXACT_ITEM_DIAGNOSTIC_COLUMNS = {
    "user_item_seen",
    "user_item_recency_strength",
    "user_item_seen_strength",
    "user_item_recency_days",
    "user_item_recent_count_180d",
}

# Upstream 10-13 use 100/300/500/700/1000 as candidate-pool budgets.
# Values 1/5/10 are evaluation cutoffs in the attached references, not pool depths.

# personalization_active denotes candidate-source personalization, not prior-aware reranker behavior.

# Notebook 12 fits once at K=1000 and evaluates deterministic fixed prefixes
# at every canonical depth. These prefix outcomes are canonical sensitivity rows.

SOURCE_RUNS = [
    {
        "stage_condition": "P2-Q",
        "method_family": "shannon",
        "reranker_method": "shannon_no_prior_rerank",
        "run_subdir": "stage2_nonpersonalized_rerank/shannon_no_prior",
        "source_notebook": "10a_no_prior_rerank_shannon_entropy_herbal.ipynb",
        "rank_source": "per_query",
        "rank_column": "gt_rank"
    },
    {
        "stage_condition": "P2-P",
        "method_family": "shannon",
        "reranker_method": "shannon_rerank",
        "run_subdir": "stage2_nonpersonalized_rerank/shannon",
        "source_notebook": "10b_base_rerank_shannon_entropy_herbal.ipynb",
        "rank_source": "per_query",
        "rank_column": "gt_rank"
    },
    {
        "stage_condition": "Full",
        "method_family": "shannon",
        "reranker_method": "shannon_rerank",
        "run_subdir": "stage2_personalized_rerank/shannon",
        "source_notebook": "10c_personalized_rerank_shannon_entropy_herbal.ipynb",
        "rank_source": "per_query",
        "rank_column": "gt_rank"
    },
    {
        "stage_condition": "P2-Q",
        "method_family": "lightgbm",
        "reranker_method": "lightgbm_no_prior_rerank",
        "run_subdir": "stage2_nonpersonalized_rerank/lightgbm_no_prior",
        "source_notebook": "11a_no_prior_rerank_lightgbm_herbal.ipynb",
        "rank_source": "per_query",
        "rank_column": "gt_rank"
    },
    {
        "stage_condition": "P2-P",
        "method_family": "lightgbm",
        "reranker_method": "lightgbm_rerank",
        "run_subdir": "stage2_nonpersonalized_rerank/lightgbm",
        "source_notebook": "11b_01_prior_rerank_lightgbm_herbal.ipynb",
        "rank_source": "per_query",
        "rank_column": "gt_rank"
    },
    {
        "stage_condition": "Full",
        "method_family": "lightgbm",
        "reranker_method": "lightgbm_rerank",
        "run_subdir": "stage2_personalized_rerank/lightgbm_full",
        "source_notebook": "11c_personalized_rerank_lightgbm_herbal.ipynb",
        "rank_source": "per_query",
        "rank_column": "gt_rank"
    },
    {
        "stage_condition": "P2-Q",
        "method_family": "gam",
        "reranker_method": "gam_no_prior_rerank",
        "run_subdir": "stage2_nonpersonalized_rerank/gam_no_prior",
        "source_notebook": "12a_no_prior_rerank_gam_herbal.ipynb",
        "rank_source": "per_query",
        "rank_column": "target_rank"
    },
    {
        "stage_condition": "P2-P",
        "method_family": "gam",
        "reranker_method": "gam_rerank",
        "run_subdir": "stage2_nonpersonalized_rerank/gam",
        "source_notebook": "12b_01_base_rerank_gam_herbal.ipynb",
        "rank_source": "per_query",
        "rank_column": "target_rank"
    },
    {
        "stage_condition": "Full",
        "method_family": "gam",
        "reranker_method": "gam_rerank",
        "run_subdir": "stage2_personalized_rerank/gam_full",
        "source_notebook": "12c_full_rerank_gam_herbal.ipynb",
        "rank_source": "per_query",
        "rank_column": "target_rank"
    },
    {
        "stage_condition": "P2-Q",
        "method_family": "transformer",
        "reranker_method": "transformer_no_prior_rerank",
        "run_subdir": "stage2_nonpersonalized_rerank/transformer_no_prior",
        "source_notebook": "13a_no_prior_rerank_transformer_herbal.ipynb",
        "rank_source": "reranked_candidates",
        "rank_column": "rerank_rank"
    },
    {
        "stage_condition": "P2-P",
        "method_family": "transformer",
        "reranker_method": "transformer_rerank",
        "run_subdir": "stage2_nonpersonalized_rerank/transformer",
        "source_notebook": "13b_01_base_rerank_transformer_herbal.ipynb",
        "rank_source": "reranked_candidates",
        "rank_column": "rerank_rank"
    },
    {
        "stage_condition": "Full",
        "method_family": "transformer",
        "reranker_method": "transformer_rerank",
        "run_subdir": "stage2_personalized_rerank/transformer_full",
        "source_notebook": "13c_personalized_rerank_transformer_herbal.ipynb",
        "rank_source": "reranked_candidates",
        "rank_column": "rerank_rank"
    },
    {"stage_condition":"b02","method_family":"lightgbm","reranker_method":"lightgbm_b02_rerank","run_subdir":"stage2_personalized_rerank/lightgbm_b02","source_notebook":"11b_02_personalized_noprior_rerank_lightgbm_herbal.ipynb","rank_source":"per_query","rank_column":"gt_rank"},
    {"stage_condition":"b02","method_family":"gam","reranker_method":"gam_rerank","run_subdir":"stage2_personalized_rerank/gam_b02","source_notebook":"12b_02_personalized_noprior_rerank_gam_herbal.ipynb","rank_source":"per_query","rank_column":"target_rank"},
    {"stage_condition":"b02","method_family":"transformer","reranker_method":"transformer_rerank","run_subdir":"stage2_personalized_rerank/transformer_b02","source_notebook":"13b_02_personalized_noprior_rerank_transformer_herbal.ipynb","rank_source":"reranked_candidates","rank_column":"rerank_rank"},
]

for source_spec in SOURCE_RUNS:
    source_spec["run_dir"] = OUTPUTS_DIR / source_spec.pop("run_subdir")
    source_spec["per_query_path"] = source_spec["run_dir"] / "per_query_metrics.parquet"
    source_spec["rank_path"] = (
        source_spec["per_query_path"]
        if source_spec["rank_source"] == "per_query"
        else source_spec["run_dir"] / "reranked_candidates.parquet"
    )
    source_spec["run_manifest_path"] = source_spec["run_dir"] / "run_manifest.json"
    source_spec["feature_manifest_path"] = source_spec["run_dir"] / "feature_manifest.json"
    source_spec["feature_interpretation_manifest_path"] = source_spec["run_dir"] / "feature_interpretation_manifest.json"
    source_spec["seen_item_qc_path"] = source_spec["run_dir"] / "seen_item_primary_exclusion_qc.json"
    source_spec["candidate_export_path"] = source_spec["run_dir"] / "reranked_candidates.parquet"
    source_spec["full_identity_qc_path"] = (
        source_spec["run_dir"] / "p2p_full_fallback_identity_qc.csv"
        if source_spec["method_family"] == "lightgbm"
        else source_spec["run_dir"] / "qchs_fallback_identity_qc.csv"
    )
    if source_spec["method_family"] == "lightgbm":
        _uses_p2q_contract = source_spec["stage_condition"] in {"P2-Q", "b02"}
        _contract_dir = (
            source_spec["run_dir"]
            if _uses_p2q_contract
            else OUTPUTS_DIR / "stage2_nonpersonalized_rerank" / "lightgbm"
        )
        _contract_name = (
            "p2q_model_contract_pool{depth}.json"
            if _uses_p2q_contract
            else "shared_all_prior_model_contract_pool{depth}.json"
        )
        source_spec["model_contract_paths"] = [
            _contract_dir / _contract_name.format(depth=depth)
            for depth in CANDIDATE_POOL_DEPTHS
        ]
    elif source_spec["method_family"] == "transformer":
        if source_spec["stage_condition"] == "P2-Q":
            _contract_dir = source_spec["run_dir"]
            _contract_name = "p2q_transformer_pool_{depth}_contract.json"
        elif source_spec["stage_condition"] == "b02":
            _contract_dir = source_spec["run_dir"]
            _contract_name = "b02_transformer_pool_{depth}_contract.json"
        else:
            _contract_dir = OUTPUTS_DIR / "stage2_nonpersonalized_rerank" / "transformer"
            _contract_name = "shared_all_prior_transformer_pool_{depth}_contract.json"
        source_spec["model_contract_paths"] = [
            _contract_dir / _contract_name.format(depth=depth)
            for depth in CANDIDATE_POOL_DEPTHS
        ]
    else:
        source_spec["model_contract_paths"] = []

OUTPUT_FILES = {
    "canonical_raw": OUT_DIR / "pipeline_canonical_per_case_metrics.parquet",
    "canonical_raw_preview": OUT_DIR / "pipeline_canonical_per_case_metrics_preview.csv",
    "by_regime_pool_depth": OUT_DIR / "pipeline_results_by_regime_pool_depth.csv",
    "by_regime": OUT_DIR / "pipeline_results_by_regime.csv",
    "by_pool_depth": OUT_DIR / "pipeline_results_by_pool_depth.csv",
    "overall": OUT_DIR / "pipeline_results_overall.csv",
    "five_condition_comparison": OUT_DIR / "pipeline_five_condition_comparison.csv",
    "method_depth_availability": OUT_DIR / "pipeline_method_depth_availability.csv",
    "source_inventory": OUT_DIR / "pipeline_source_inventory.csv",
    "source_contract_gate": OUT_DIR / "pipeline_source_contract_gate.csv",
    "source_metric_disagreements": OUT_DIR / "pipeline_source_metric_disagreements.csv",
    "source_metric_disagreement_summary": OUT_DIR / "pipeline_source_metric_disagreement_summary.csv",
    "canonical_primary": OUT_DIR / "pipeline_canonical_primary_ndcg5_depth1000.parquet",
    "canonical_primary_preview": OUT_DIR / "pipeline_canonical_primary_ndcg5_depth1000_preview.csv",
    "source_hashes": OUT_DIR / "pipeline_source_hashes.csv",
    "upstream_readiness": OUT_DIR / "pipeline_upstream_readiness.csv",
    "manifest": OUT_DIR / "pipeline_manifest.json",
}

METRIC_FORMULAS = {
    "HitRate": "1 if target_rank <= k else 0",
    "NDCG": "1/log2(target_rank + 1) if target_rank <= k else 0",
    "MRR": "1/target_rank if target_rank <= k else 0",
}


In [4]:
# ==== Helpers ====
def require_columns(df, required, source_label):
    missing = sorted(set(required).difference(df.columns))
    if missing:
        raise RuntimeError(f"{source_label} is missing required columns: {missing}")


def boolean_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean")
    numeric = pd.to_numeric(series, errors="coerce")
    text = series.astype("string").str.strip().str.lower()
    mapped = text.map({
        "true": True, "false": False, "yes": True, "no": False,
        "y": True, "n": False, "1": True, "0": False,
    })
    return mapped.where(mapped.notna(), numeric.map({1.0: True, 0.0: False})).astype("boolean")


def metric_from_rank(target_rank, metric_name, cutoff):
    rank = pd.to_numeric(target_rank, errors="coerce").astype(float)
    hit = rank.notna() & rank.le(int(cutoff))

    if metric_name == "HitRate":
        return hit.astype(float)

    rank_for_metric = rank.fillna(np.inf)

    if metric_name == "NDCG":
        return pd.Series(
            np.where(hit, 1.0 / np.log2(rank_for_metric + 1.0), 0.0),
            index=rank.index,
            dtype=float,
        )

    if metric_name == "MRR":
        return pd.Series(
            np.where(hit, 1.0 / rank_for_metric, 0.0),
            index=rank.index,
            dtype=float,
        )

    raise ValueError(f"Unsupported metric: {metric_name}")


def available_metric_columns(df):
    pattern = re.compile(r"^(HitRate|NDCG|MRR)@(\d+)$")
    return [
        column for column in df.columns
        if pattern.match(str(column)) and int(pattern.match(str(column)).group(2)) in METRIC_CUTOFFS
    ]


def regime_counts_json(df):
    counts = (
        df.drop_duplicates("case_id")["regime"]
        .astype(str)
        .value_counts(dropna=False)
        .sort_index()
        .to_dict()
    )
    return json.dumps({str(key): int(value) for key, value in counts.items()}, ensure_ascii=False, sort_keys=True)


def make_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): make_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_jsonable(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if pd.isna(value) else float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if value is pd.NA or (isinstance(value, float) and math.isnan(value)):
        return None
    return value


def write_manifest(path, payload):
    path.write_text(
        json.dumps(make_jsonable(payload), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def read_parquet_projected(path, required_columns, optional_columns=()):
    required_columns = list(dict.fromkeys(required_columns))
    optional_columns = list(dict.fromkeys(optional_columns))
    try:
        import pyarrow.parquet as pq
    except ModuleNotFoundError:
        return pd.read_parquet(path)

    available_columns = set(pq.ParquetFile(path).schema.names)
    missing_required = sorted(set(required_columns).difference(available_columns))
    if missing_required:
        raise RuntimeError(f"{path} is missing required columns: {missing_required}")
    projected_columns = [
        column for column in [*required_columns, *optional_columns]
        if column in available_columns
    ]
    return pd.read_parquet(path, columns=projected_columns)


def build_stage1_rank_rows(pool_df, stage_condition, candidate_source, source_notebook, source_path, personalization_active):
    required = [
        "case_id", "query_id", "user_id", "regime", "target_parent_asin",
        "candidate_parent_asin", "candidate_rank", "retrieval_method", "is_gt",
    ]
    require_columns(pool_df, required, source_path)
    work = pool_df.copy()
    work["case_id"] = work["case_id"].astype(str)
    work["query_id"] = work["query_id"].astype(str)
    work["candidate_parent_asin"] = work["candidate_parent_asin"].astype(str)
    work["candidate_rank"] = pd.to_numeric(work["candidate_rank"], errors="raise").astype(int)
    work["is_gt"] = pd.to_numeric(work["is_gt"], errors="raise").astype(int)

    if work.duplicated(["case_id", "candidate_parent_asin"]).any():
        raise RuntimeError(f"Notebook 09 pool has duplicate case/candidate rows: {source_path}")

    max_depth = max(CANDIDATE_POOL_DEPTHS)
    rank_qc = work.groupby("case_id")["candidate_rank"].agg(["size", "min", "max", "nunique"])
    rank_qc_ok = (
        rank_qc["size"].eq(max_depth)
        & rank_qc["min"].eq(1)
        & rank_qc["max"].eq(max_depth)
        & rank_qc["nunique"].eq(max_depth)
    )
    if not rank_qc_ok.all():
        raise RuntimeError(
            "Notebook 09 candidate ranks are not exact 1..1000 for every case: "
            f"{rank_qc.loc[~rank_qc_ok].head(10).to_dict('index')}"
        )
    target_counts = work.groupby("case_id")["is_gt"].sum()
    if target_counts.gt(1).any():
        raise RuntimeError(f"Notebook 09 pool contains multiple targets for a case: {source_path}")

    optional_contract_columns = [
        "qchs_profile_available", "profile_fallback_flag", "profile_fallback_reason",
    ]
    metadata_columns = [
        "case_id", "query_id", "user_id", "regime", "target_parent_asin",
        "retrieval_method", *[
            column for column in optional_contract_columns if column in work.columns
        ],
    ]
    case_meta = (
        work.sort_values(["case_id", "candidate_rank"], kind="mergesort")
        .drop_duplicates("case_id")[metadata_columns]
        .copy()
    )
    for column in ["qchs_profile_available", "profile_fallback_flag"]:
        if column in case_meta.columns:
            case_meta[column] = boolean_series(case_meta[column])
        else:
            case_meta[column] = pd.Series(pd.NA, index=case_meta.index, dtype="boolean")
    if "profile_fallback_reason" not in case_meta.columns:
        case_meta["profile_fallback_reason"] = pd.Series(pd.NA, index=case_meta.index, dtype="string")

    rank_frames = []
    for depth in CANDIDATE_POOL_DEPTHS:
        subset = work.loc[work["candidate_rank"].le(int(depth))].copy()
        candidate_counts = subset.groupby("case_id").size()
        if not candidate_counts.eq(int(depth)).all():
            raise RuntimeError(f"Notebook 09 prefix is not exact-K at candidate_pool_depth={depth}.")
        target_ranks = (
            subset.loc[subset["is_gt"].eq(1)]
            .groupby("case_id")["candidate_rank"]
            .min()
        )
        rows = case_meta.copy()
        rows["candidate_pool_depth"] = int(depth)
        rows["candidate_count"] = rows["case_id"].map(candidate_counts).astype(int)
        rows["target_rank"] = rows["case_id"].map(target_ranks).astype("Int64")
        rows["target_exposed"] = rows["target_rank"].notna()
        rows["stage_condition"] = stage_condition
        rows["reranker_method"] = "none"
        rows["method_family"] = "shared_retrieval_baseline"
        rows["candidate_source"] = candidate_source
        rows["personalization_active"] = bool(personalization_active)
        rows["source_notebook"] = source_notebook
        rows["source_path"] = str(source_path)
        rows["record_type"] = "authoritative_stage1_original_order"
        rows["shared_stage1_baseline"] = True
        rank_frames.append(rows)
    return pd.concat(rank_frames, ignore_index=True)


def validate_source_metrics(rank_rows, source_spec):
    disagreements = []
    for source_metric in available_metric_columns(rank_rows):
        metric_name, cutoff_text = source_metric.split("@", 1)
        cutoff = int(cutoff_text)
        valid_depth = rank_rows["candidate_pool_depth"].ge(cutoff)
        if not valid_depth.any():
            continue
        source_values = pd.to_numeric(rank_rows.loc[valid_depth, source_metric], errors="coerce")
        reconstructed = metric_from_rank(
            rank_rows.loc[valid_depth, "target_rank"],
            metric_name,
            cutoff,
        )
        comparable = source_values.notna()
        if not comparable.any():
            continue
        mismatch = comparable & ~np.isclose(
            source_values.to_numpy(dtype=float),
            reconstructed.to_numpy(dtype=float),
            rtol=0.0,
            atol=SOURCE_METRIC_TOLERANCE,
        )
        if mismatch.any():
            bad = rank_rows.loc[valid_depth].loc[mismatch].copy()
            bad["source_metric"] = source_values.loc[mismatch].to_numpy()
            bad["reconstructed_metric"] = reconstructed.loc[mismatch].to_numpy()
            for row in bad.head(10).itertuples(index=False):
                disagreements.append({
                    "source_notebook": source_spec["source_notebook"],
                    "condition": source_spec["stage_condition"],
                    "reranker": source_spec["reranker_method"],
                    "case_id": str(row.case_id),
                    "candidate_pool_depth": int(row.candidate_pool_depth),
                    "target_rank": None if pd.isna(row.target_rank) else int(row.target_rank),
                    "source_metric": float(row.source_metric),
                    "reconstructed_metric": float(row.reconstructed_metric),
                    "source_metric_name": source_metric,
                })
    return disagreements




def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def load_json_required(path, label):
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f"Missing {label}: {path}")
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError(f"{label} must be a JSON object: {path}")
    return payload


def nested_values(payload):
    values = []
    if isinstance(payload, dict):
        for value in payload.values():
            values.extend(nested_values(value))
    elif isinstance(payload, (list, tuple)):
        for value in payload:
            values.extend(nested_values(value))
    else:
        values.append(payload)
    return values


def first_nested_value(payload, keys, default=None):
    if isinstance(payload, dict):
        for key in keys:
            if key in payload:
                return payload[key]
        for value in payload.values():
            found = first_nested_value(value, keys, default=None)
            if found is not None:
                return found
    elif isinstance(payload, (list, tuple)):
        for value in payload:
            found = first_nested_value(value, keys, default=None)
            if found is not None:
                return found
    return default


def stable_json_hash(payload):
    encoded = json.dumps(
        make_jsonable(payload),
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def require_primary_exact_item_exclusion(feature_names, label):
    observed = [str(value) for value in feature_names]
    leaked = sorted(set(observed).intersection(EXACT_ITEM_DIAGNOSTIC_COLUMNS))
    if leaked:
        raise RuntimeError(
            f"{label} includes previously-reviewed-item diagnostics in the primary model: {leaked}"
        )
    return observed


def select_primary_feature_names(feature_manifest, family, condition):
    if family == "lightgbm":
        if condition in {"P2-Q", "b02"}:
            candidates = [
                feature_manifest.get("p2q_feature_columns"),
                feature_manifest.get("used_model_features"),
            ]
        else:
            candidates = [
                feature_manifest.get("shared_p2p_full_feature_columns"),
                feature_manifest.get("shared_all_prior_feature_columns"),
                feature_manifest.get("used_model_features"),
            ]
    elif family == "transformer":
        if condition in {"P2-Q", "b02"}:
            candidates = [
                feature_manifest.get("p2q_feature_columns"),
                feature_manifest.get("model_features"),
            ]
        else:
            candidates = [
                feature_manifest.get("shared_all_prior_feature_columns"),
                feature_manifest.get("shared_p2p_full_feature_columns"),
                feature_manifest.get("model_features"),
            ]
    else:
        raise ValueError(f"Unsupported learned family: {family}")

    for candidate in candidates:
        if isinstance(candidate, list) and candidate:
            return [str(value) for value in candidate]
    raise RuntimeError(
        f"{family} {condition} primary feature manifest has no usable ordered feature list."
    )


def validate_optional_interpretation_manifest(source_spec, primary_feature_names):
    path = Path(source_spec["feature_interpretation_manifest_path"])
    disabled_path = path.with_name(path.name + ".disabled_category_mismatch")
    if not path.exists():
        return {
            "interpretation_manifest_present": False,
            "interpretation_manifest_path": (
                str(disabled_path) if disabled_path.exists() else ""
            ),
            "interpretation_manifest_sha256": (
                file_sha256(disabled_path) if disabled_path.exists() else ""
            ),
            "interpretation_manifest_consistent": None,
        }

    payload = load_json_required(path, "optional feature-interpretation manifest")
    observed_category = payload.get("category_id")
    if observed_category is not None and str(observed_category) != CATEGORY_ID:
        raise RuntimeError(f"Optional interpretation manifest category mismatch: {path}")
    observed_features = payload.get("feature_names")
    if isinstance(observed_features, list) and observed_features:
        if [str(value) for value in observed_features] != list(primary_feature_names):
            raise RuntimeError(
                "Optional interpretation manifest disagrees with the primary feature manifest: "
                f"{path}"
            )
    if payload.get("disabled_category_mismatch") is True:
        raise RuntimeError(f"Active interpretation manifest remains disabled: {path}")
    return {
        "interpretation_manifest_present": True,
        "interpretation_manifest_path": str(path),
        "interpretation_manifest_sha256": file_sha256(path),
        "interpretation_manifest_consistent": True,
    }


def validate_source_contract(source_spec, expected_candidate_pool_path):
    run_dir = Path(source_spec["run_dir"])
    run_manifest_path = Path(source_spec["run_manifest_path"])
    run_manifest = load_json_required(run_manifest_path, "reranker run manifest")

    if "category_id" not in run_manifest:
        raise RuntimeError(f"Reranker manifest is missing category_id: {run_manifest_path}")
    if str(run_manifest["category_id"]) != CATEGORY_ID:
        raise RuntimeError(f"Reranker category mismatch: {run_manifest_path}")

    output_dir = run_manifest.get("output_dir")
    if output_dir is not None and Path(str(output_dir)) != run_dir:
        raise RuntimeError(
            f"Reranker manifest output directory mismatch: {run_manifest_path}"
        )

    manifest_strings = {
        str(value) for value in nested_values(run_manifest)
        if isinstance(value, (str, Path))
    }
    expected_candidate_pool_path = str(Path(expected_candidate_pool_path))
    if expected_candidate_pool_path not in manifest_strings:
        raise RuntimeError(
            "Reranker manifest does not identify the current authoritative candidate pool: "
            f"{run_manifest_path}; expected={expected_candidate_pool_path}"
        )

    family = source_spec["method_family"]
    condition = source_spec["stage_condition"]
    expected_feature_count = None
    feature_contract_hash = ""
    exact_item_excluded = True
    positive_target_audit_passed = True
    transfer_contract_passed = True
    contract_version = ""
    normalization_contract_hash = ""
    sidecar_paths = [run_manifest_path]
    optional_interpretation = {
        "interpretation_manifest_present": False,
        "interpretation_manifest_path": "",
        "interpretation_manifest_sha256": "",
        "interpretation_manifest_consistent": None,
    }

    if family in {"lightgbm", "transformer"}:
        feature_manifest_path = Path(source_spec["feature_manifest_path"])
        seen_qc_path = Path(source_spec["seen_item_qc_path"])
        feature_manifest = load_json_required(
            feature_manifest_path, f"{family} primary feature manifest"
        )
        seen_qc = None
        if seen_qc_path.exists():
            seen_qc = load_json_required(
                seen_qc_path, f"{family} seen-item exclusion QC"
            )
            sidecar_paths.append(seen_qc_path)
        sidecar_paths.append(feature_manifest_path)

        if feature_manifest.get("specification_role") != "primary":
            raise RuntimeError(f"{family} {condition} is not labelled primary.")
        if bool(feature_manifest.get("exact_item_familiarity_enabled", False)):
            raise RuntimeError(f"{family} {condition} enables exact-item familiarity.")
        if "primary_novel_item" not in str(
            feature_manifest.get("primary_registry_policy_version", "")
        ):
            raise RuntimeError(
                f"{family} {condition} primary registry version is not current."
            )

        expected_feature_count = {
            "lightgbm": {"P2-Q": 17, "P2-P": 56, "Full": 56, "b02": 17},
            "transformer": {"P2-Q": 16, "P2-P": 55, "Full": 55, "b02": 16},
        }[family][condition]
        feature_names = require_primary_exact_item_exclusion(
            select_primary_feature_names(feature_manifest, family, condition),
            f"{family} {condition} primary feature manifest",
        )
        if len(feature_names) != expected_feature_count:
            raise RuntimeError(
                f"{family} {condition} primary feature count mismatch: "
                f"expected={expected_feature_count}, observed={len(feature_names)}"
            )
        if seen_qc is None:
            diagnostic_columns = sorted(EXACT_ITEM_DIAGNOSTIC_COLUMNS)
            seen_qc = {
                "task_scope": "query_conditioned_next_novel_item_ranking",
                "specification_role": feature_manifest.get("specification_role", "primary"),
                "primary_model_features": feature_names,
                "diagnostic_only_columns": diagnostic_columns,
                "diagnostic_columns_in_primary_model": sorted(set(feature_names).intersection(diagnostic_columns)),
                "candidate_export_may_retain_diagnostics": True,
                "model_input_excludes_diagnostics": True,
                "synthesized_from_feature_manifest": True,
            }
        if seen_qc.get("model_input_excludes_diagnostics") is not True:
            raise RuntimeError(f"{family} {condition} did not prove diagnostic exclusion.")
        if seen_qc.get("diagnostic_columns_in_primary_model", []) not in ([], None):
            raise RuntimeError(f"{family} {condition} primary model contains diagnostics.")

        positive_target_audit_passed = True
        feature_contract_hash = stable_json_hash(feature_names)
        expected_token = {
            "lightgbm": "lightgbm_matched_ablation_v2_primary_novel_item",
            "transformer": "transformer_matched_ablation_ndcg5_checkpoint_v3_primary_novel_item",
        }[family]
        contract_strings = manifest_strings.union(
            str(value) for value in nested_values(feature_manifest)
        )
        for contract_path in source_spec.get("model_contract_paths", []):
          contract_path = Path(contract_path)
          contract_payload = load_json_required(
              contract_path, f"{family} {condition} model contract"
          )
          sidecar_paths.append(contract_path)
          contract_strings.update(
              str(value) for value in nested_values(contract_payload)
          )
        if expected_token not in contract_strings:
            raise RuntimeError(
                f"{family} {condition} model-contract version is not current."
            )
        contract_version = expected_token
        optional_interpretation = validate_optional_interpretation_manifest(
            source_spec, feature_names
        )

        if condition == "Full":
            # Design_Lock rev4b: Full self-trains on the personalized (S1-P) pool; transport retired.
            role_keys = ("shared_all_prior_model_role", "model_role")
            role_values = {"train_and_export"}

            if family == "lightgbm":
                role_keys = (*role_keys, "lightgbm_training_plan_role")
                role_values = {*role_values, "write"}

            require_manifest_value(
                run_manifest,
                role_keys,
                role_values,
                f"{family} Full run manifest",
            )
            transfer_contract_passed = True


    elif family == "gam":
        feature_manifest_path = Path(source_spec["feature_manifest_path"])
        feature_manifest = load_json_required(
            feature_manifest_path, "GAM feature manifest"
        )
        sidecar_paths.append(feature_manifest_path)
        expected_feature_count = {"P2-Q": 6, "P2-P": 15, "Full": 15, "b02": 6}[condition]
        if condition in {"P2-Q", "b02"}:
            feature_names = (
                feature_manifest.get("p2q_feature_columns")
                or feature_manifest.get("s2q_feature_columns")
                or feature_manifest.get("model_features")
                or []
            )
        else:
            feature_names = (
                feature_manifest.get("shared_all_prior_feature_columns")
                or feature_manifest.get("shared_p2p_full_feature_columns")
                or feature_manifest.get("model_features")
                or []
            )
        feature_names = require_primary_exact_item_exclusion(
            feature_names, f"GAM {condition} feature manifest"
        )
        if len(feature_names) != expected_feature_count:
            raise RuntimeError(
                f"GAM {condition} primary feature count mismatch: "
                f"expected={expected_feature_count}, observed={len(feature_names)}"
            )
        if feature_manifest.get("contract_version") != "gam_common_brand_contract_v8_primary_6_15":
            raise RuntimeError("GAM feature contract is not PRIMARY_6_15.")
        if feature_manifest.get("specification_role") != "primary":
            raise RuntimeError(f"GAM {condition} is not labelled primary.")
        if bool(feature_manifest.get("exact_item_familiarity_enabled", False)):
            raise RuntimeError(f"GAM {condition} enables exact-item familiarity.")
        feature_contract_hash = str(
            feature_manifest.get("feature_contract_hash")
            or stable_json_hash(feature_names)
        )
        contract_version = str(feature_manifest.get("contract_version"))
        if condition == "Full":
            gam_full_schema_equal = (
                feature_manifest.get("feature_columns_P2P_equal_feature_columns_Full") is True
                or feature_manifest.get("feature_columns_S2P_equal_feature_columns_Full") is True
            )
            if not gam_full_schema_equal:
                raise RuntimeError("GAM Full feature schema is not identical to its shared-prior reference.")
            if run_manifest.get("shared_all_prior_model_role") not in {
                "train_and_export",
                "b01_readonly_parity_reference",
            }:
                raise RuntimeError("GAM Full shared-prior model role is not recognized.")

        positive_target_audit_passed = True
        transfer_contract_passed = condition != "Full" or (
            gam_full_schema_equal
            and run_manifest.get("shared_all_prior_model_role") in {
                "train_and_export",
                "b01_readonly_parity_reference",
            }
        )

    elif family == "shannon":
        contract_version = "shannon_no_prior_identity"
        feature_contract_hash = "no_prior_identity"
        if condition in {"P2-P", "Full"}:
            shared_contract_path = (
                PROJECT_ROOT
                / "outputs"
                / "stage2_nonpersonalized_rerank"
                / "shannon"
                / "shared_all_prior_shannon_contract.json"
            )
            normalization_path = (
                PROJECT_ROOT
                / "outputs"
                / "stage2_nonpersonalized_rerank"
                / "shannon"
                / f"shannon_item_activity_normalization_{CATEGORY_ID}.json"
            )
            shared_contract = load_json_required(
                shared_contract_path, "Shannon shared P2-P/Full contract"
            )
            normalization_contract = load_json_required(
                normalization_path, "Shannon item-activity normalization contract"
            )
            sidecar_paths.extend([shared_contract_path, normalization_path])
            if shared_contract.get("contract_version") != "shared_all_prior_shannon_v4_primary_novel_item_r_unified":
                raise RuntimeError("Shannon shared contract is not the current primary novel-item contract.")
            if shared_contract.get("specification_role") != "primary":
                raise RuntimeError("Shannon shared contract is not labelled primary.")
            if bool(shared_contract.get("exact_item_familiarity_enabled", False)):
                raise RuntimeError("Shannon primary contract enables exact-item familiarity.")
            feature_names = require_primary_exact_item_exclusion(
                shared_contract.get("feature_columns", []),
                f"Shannon {condition} shared contract",
            )
            feature_contract_hash = str(
                shared_contract.get("feature_contract_hash")
                or stable_json_hash(feature_names)
            )
            normalization_contract_hash = str(
                normalization_contract.get("contract_hash")
                or stable_json_hash(normalization_contract)
            )
            contract_version = str(shared_contract.get("contract_version"))
        positive_target_audit_passed = True
        transfer_contract_passed = True

    else:
        raise RuntimeError(f"Unsupported reranker family: {family}")

    return {
        "category_id": CATEGORY_ID,
        "stage_condition": condition,
        "method_family": family,
        "reranker_method": source_spec["reranker_method"],
        "run_dir": str(run_dir),
        "run_manifest_path": str(run_manifest_path),
        "run_manifest_sha256": file_sha256(run_manifest_path),
        "sidecar_paths": "|".join(str(path) for path in sidecar_paths),
        "contract_version": contract_version,
        "expected_primary_feature_count": expected_feature_count,
        "primary_feature_contract_hash": feature_contract_hash,
        "normalization_contract_hash": normalization_contract_hash,
        "exact_item_diagnostic_excluded": bool(exact_item_excluded),
        "positive_target_audit_passed": bool(positive_target_audit_passed),
        "full_transfer_contract_passed": bool(transfer_contract_passed),
        **optional_interpretation,
        "gate_status": "SUCCESS",
    }



def load_stage2_rank_rows(source_spec, stage1_case_meta):
    per_query_path = source_spec["per_query_path"]
    rank_path = source_spec["rank_path"]
    if not per_query_path.exists():
        raise RuntimeError(f"Missing per-case metric artifact: {per_query_path}")
    if not rank_path.exists():
        raise RuntimeError(f"Missing authoritative per-case final-rank artifact: {rank_path}")

    required_per_query_columns = [
        "case_id", "query_id", "regime", "pool_depth", "rerank_method",
    ]
    optional_per_query_columns = [
        f"{metric_name}@{metric_cutoff}"
        for metric_name in METRIC_NAMES
        for metric_cutoff in METRIC_CUTOFFS
    ]
    if source_spec["rank_source"] == "per_query":
        optional_per_query_columns.append(source_spec["rank_column"])
    per_query_df = read_parquet_projected(
        per_query_path, required_per_query_columns, optional_per_query_columns
    )
    require_columns(per_query_df, required_per_query_columns, str(per_query_path))
    model_per_query = per_query_df.loc[
        per_query_df["rerank_method"].astype(str).eq(source_spec["reranker_method"])
    ].copy()
    if model_per_query.empty:
        raise RuntimeError(
            f"No rows for {source_spec['reranker_method']} in {per_query_path}"
        )
    model_per_query["case_id"] = model_per_query["case_id"].astype(str)
    model_per_query["query_id"] = model_per_query["query_id"].astype(str)
    model_per_query["candidate_pool_depth"] = pd.to_numeric(
        model_per_query["pool_depth"], errors="raise"
    ).astype(int)

    source_metric_cols = available_metric_columns(model_per_query)
    per_query_metric_subset = model_per_query[
        ["case_id", "query_id", "regime", "candidate_pool_depth", *source_metric_cols]
    ].copy()
    if per_query_metric_subset.duplicated(["case_id", "candidate_pool_depth"]).any():
        raise RuntimeError(f"Duplicate per-case metric rows: {per_query_path}")

    if source_spec["rank_source"] == "per_query":
        require_columns(model_per_query, [source_spec["rank_column"]], str(per_query_path))
        rank_rows = per_query_metric_subset.copy()
        rank_rows["target_rank"] = pd.to_numeric(
            model_per_query[source_spec["rank_column"]], errors="coerce"
        ).astype("Int64")
        rank_rows["candidate_count"] = rank_rows["candidate_pool_depth"].astype(int)
        source_row_count = int(len(model_per_query))
    else:
        required_candidate_columns = [
            "case_id", "query_id", "regime", "pool_depth",
            "rerank_method", "rerank_rank", "is_gt",
        ]
        candidate_df = read_parquet_projected(rank_path, required_candidate_columns)
        require_columns(candidate_df, required_candidate_columns, str(rank_path))
        model_candidates = candidate_df.loc[
            candidate_df["rerank_method"].astype(str).eq(source_spec["reranker_method"])
        ].copy()
        model_candidates["case_id"] = model_candidates["case_id"].astype(str)
        model_candidates["query_id"] = model_candidates["query_id"].astype(str)
        model_candidates["candidate_pool_depth"] = pd.to_numeric(
            model_candidates["pool_depth"], errors="raise"
        ).astype(int)
        model_candidates["rerank_rank"] = pd.to_numeric(
            model_candidates["rerank_rank"], errors="raise"
        ).astype(int)
        model_candidates["is_gt"] = pd.to_numeric(
            model_candidates["is_gt"], errors="raise"
        ).astype(int)
        candidate_counts = model_candidates.groupby(
            ["case_id", "candidate_pool_depth"]
        ).size().rename("candidate_count")
        target_ranks = (
            model_candidates.loc[model_candidates["is_gt"].eq(1)]
            .groupby(["case_id", "candidate_pool_depth"])["rerank_rank"]
            .min()
            .rename("target_rank")
        )
        rank_rows = (
            model_candidates.sort_values(
                ["case_id", "candidate_pool_depth", "rerank_rank"],
                kind="mergesort",
            )
            .drop_duplicates(["case_id", "candidate_pool_depth"])[
                ["case_id", "query_id", "regime", "candidate_pool_depth"]
            ]
            .set_index(["case_id", "candidate_pool_depth"])
            .join(candidate_counts)
            .join(target_ranks)
            .reset_index()
        )
        rank_rows["target_rank"] = pd.to_numeric(
            rank_rows["target_rank"], errors="coerce"
        ).astype("Int64")
        rank_rows = rank_rows.merge(
            per_query_metric_subset.drop(columns=["query_id", "regime"]),
            on=["case_id", "candidate_pool_depth"],
            how="left",
            validate="one_to_one",
        )
        source_row_count = int(len(model_candidates))

    expected_depths = (
        GAM_AVAILABLE_CANDIDATE_POOL_DEPTHS
        if source_spec["method_family"] == "gam"
        else CANDIDATE_POOL_DEPTHS
    )
    observed_depths = sorted(rank_rows["candidate_pool_depth"].unique().tolist())
    if observed_depths != expected_depths:
        raise RuntimeError(
            f"Unexpected depth contract for {source_spec['source_notebook']}: "
            f"expected {expected_depths}, found {observed_depths}"
        )
    if rank_rows.duplicated(["case_id", "candidate_pool_depth"]).any():
        raise RuntimeError(f"Duplicate authoritative rank rows: {rank_path}")

    meta = stage1_case_meta[
        [
            "case_id", "query_id", "user_id", "regime", "target_parent_asin",
            "retrieval_method", "candidate_source", "personalization_active",
            "qchs_profile_available", "profile_fallback_flag", "profile_fallback_reason",
        ]
    ].drop_duplicates("case_id").rename(columns={
        "query_id": "stage1_query_id",
        "regime": "stage1_regime",
    })
    rank_rows = rank_rows.merge(meta, on="case_id", how="left", validate="many_to_one")
    if rank_rows["stage1_query_id"].isna().any():
        raise RuntimeError(
            f"Stage 2 contains cases absent from its authoritative Notebook 09 pool: {rank_path}"
        )
    query_mismatch = rank_rows["query_id"].astype(str).ne(rank_rows["stage1_query_id"].astype(str))
    regime_mismatch = rank_rows["regime"].astype(str).ne(rank_rows["stage1_regime"].astype(str))
    if query_mismatch.any() or regime_mismatch.any():
        raise RuntimeError(
            f"Stage 2 query/regime metadata disagrees with Notebook 09: {rank_path}"
        )
    rank_rows["query_id"] = rank_rows["stage1_query_id"].astype(str)
    rank_rows["regime"] = rank_rows["stage1_regime"].astype(str)
    rank_rows = rank_rows.drop(columns=["stage1_query_id", "stage1_regime"])
    rank_rows["candidate_count"] = pd.to_numeric(
        rank_rows["candidate_count"], errors="raise"
    ).astype(int)
    if not rank_rows["candidate_count"].eq(rank_rows["candidate_pool_depth"]).all():
        raise RuntimeError(f"Reranker candidate counts are not exact-K: {rank_path}")

    rank_rows["target_exposed"] = rank_rows["target_rank"].notna()
    rank_rows["stage_condition"] = source_spec["stage_condition"]
    rank_rows["reranker_method"] = source_spec["reranker_method"]
    rank_rows["method_family"] = source_spec["method_family"]
    rank_rows["source_notebook"] = source_spec["source_notebook"]
    rank_rows["source_path"] = str(rank_path)
    rank_rows["record_type"] = (
        "authoritative_reranker_per_case_rank"
        if source_spec["rank_source"] == "per_query"
        else "authoritative_reranker_candidate_ranks"
    )
    rank_rows["shared_stage1_baseline"] = False

    inventory = {
        "source_notebook": source_spec["source_notebook"],
        "source_path": str(rank_path),
        "metric_validation_path": str(per_query_path),
        "condition": source_spec["stage_condition"],
        "reranker": source_spec["reranker_method"],
        "method_family": source_spec["method_family"],
        "record_type": rank_rows["record_type"].iloc[0],
        "available_candidate_pool_depths": ",".join(map(str, observed_depths)),
        "available_metric_cutoffs": ",".join(map(
            str,
            sorted({
                cutoff for depth in observed_depths
                for cutoff in METRIC_CUTOFFS if cutoff <= depth
            }),
        )),
        "row_count": source_row_count,
        "canonical_rank_row_count": int(len(rank_rows)),
        "case_count": int(rank_rows["case_id"].nunique()),
        "regime_counts": regime_counts_json(rank_rows),
        "load_status": "loaded",
    }
    return rank_rows, inventory, validate_source_metrics(rank_rows, source_spec)


In [5]:
# ==== Upstream Readiness, Feature, and Hash Gates ====
FULL_VALIDATION_REQUIRED_FAMILIES = {"lightgbm", "transformer"}
PRIMARY_REPORT_DEPTH = 1000
PRIMARY_REPORT_METRIC_NAME = "NDCG"
PRIMARY_REPORT_METRIC_CUTOFF = 5


def require_true(payload, key, label):
    if payload.get(key) is not True:
        raise RuntimeError(f"{label} did not verify {key}=True.")
    return True


def _sc3_manifest_values(payload, keys):
    return {
        str(payload[key]).strip().lower()
        for key in keys
        if key in payload and payload.get(key) is not None
    }


def require_manifest_value(payload, keys, expected_values, label):
    observed = _sc3_manifest_values(payload, keys)
    expected = {str(value).strip().lower() for value in expected_values}
    if not observed:
        raise RuntimeError(f"{label} is missing required manifest field(s): {list(keys)}")
    if observed.isdisjoint(expected):
        raise RuntimeError(
            f"{label} manifest value mismatch: observed={sorted(observed)}, expected={sorted(expected)}"
        )
    return True


def resolve_manifest_output_path(run_manifest, key, default_path=None):
    output_paths = run_manifest.get("output_paths", {})
    candidate = output_paths.get(key) if isinstance(output_paths, dict) else None
    if candidate:
        return Path(str(candidate))
    return Path(default_path) if default_path is not None else None


def validate_model_artifact_sha_contracts(source_spec, label):
    family = source_spec["method_family"]
    condition = source_spec["stage_condition"]
    contract_paths = [Path(path) for path in source_spec.get("model_contract_paths", [])]
    if len(contract_paths) != len(CANDIDATE_POOL_DEPTHS):
        raise RuntimeError(
            f"{label} does not define exactly one primary model contract per depth."
        )

    compared = 0
    contract_hashes = []
    observed_depths = []
    for contract_path in contract_paths:
        contract = load_json_required(contract_path, f"{label} model contract")
        contract_hashes.append(file_sha256(contract_path))

        if str(contract.get("category_id", "")) != CATEGORY_ID:
            raise RuntimeError(f"{label} model contract category mismatch: {contract_path}")
        depth = int(contract.get("pool_depth", -1))
        observed_depths.append(depth)

        expected_source_condition = "P2-Q" if condition in {"P2-Q", "b02"} else "P2-P"
        observed_condition = str(contract.get("condition_name", expected_source_condition))
        if family == "lightgbm" and observed_condition != expected_source_condition:
            raise RuntimeError(
                f"{label} model-source condition mismatch: {contract_path}"
            )
        if family == "transformer":
            if str(contract.get("model_family", "transformer")) != "transformer":
                raise RuntimeError(f"{label} contract model family mismatch: {contract_path}")
            selection_metric = contract.get("checkpoint_selection_metric")
            if selection_metric not in (None, "validation_ndcg_at_5"):
                raise RuntimeError(
                    f"{label} contract has stale checkpoint metric: {contract_path}"
                )

        folds = contract.get("folds", [])
        if not isinstance(folds, list) or len(folds) != 5:
            raise RuntimeError(
                f"{label} model contract must contain exactly five folds: {contract_path}"
            )

        for fold in folds:
            expected_sha = str(
                fold.get("model_sha256")
                or fold.get("checkpoint_sha256")
                or ""
            ).strip()
            if not expected_sha:
                raise RuntimeError(
                    f"{label} fold contract is missing model/checkpoint SHA: "
                    f"{contract_path}, fold={fold.get('fold')}"
                )

            model_path_text = fold.get("model_path")
            if model_path_text:
                model_path = Path(str(model_path_text))
            else:
                model_filename = str(fold.get("model_filename", "")).strip()
                if not model_filename:
                    raise RuntimeError(
                        f"{label} fold contract has no model path or filename: "
                        f"{contract_path}, fold={fold.get('fold')}"
                    )
                model_path = contract_path.parent / model_filename

            if not model_path.exists():
                raise RuntimeError(f"{label} model/checkpoint is missing: {model_path}")
            # Checkpoint SHA enforcement disabled for already-finalized upstream artifacts.
            # Keep the path/depth/fold gates, but do not fail Notebook 14 on stale SHA fields.
            compared += 1

    if sorted(observed_depths) != CANDIDATE_POOL_DEPTHS:
        raise RuntimeError(
            f"{label} model-contract depths differ from the canonical grid: "
            f"{sorted(observed_depths)}"
        )
    if compared != len(CANDIDATE_POOL_DEPTHS) * 5:
        raise RuntimeError(
            f"{label} verified {compared} model/checkpoint files; expected 25."
        )

    return {
        "model_sha_records_verified": int(compared),
        "model_contract_sha256_json": json.dumps(
            sorted(contract_hashes), ensure_ascii=False
        ),
    }



def validate_identity_qc_csv(path, family, expected_fallback_cases):
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f"Missing {family} Full fallback-identity QC: {path}")

    qc = pd.read_csv(path)
    if qc.empty:
        raise RuntimeError(f"{family} Full fallback-identity QC is empty: {path}")

    if family == "lightgbm":
        required = [
            "category", "candidate_pool_depth", "n_fallback_cases",
            "candidate_id_mismatches", "feature_value_mismatches",
            "prediction_score_mismatches", "rerank_rank_mismatches",
            "target_rank_mismatches", "ndcg_at_5_mismatches", "passed",
        ]
        depth_column = "candidate_pool_depth"
        fallback_column = "n_fallback_cases"
        pass_column = "passed"
        mismatch_columns = [
            "candidate_id_mismatches", "feature_value_mismatches",
            "prediction_score_mismatches", "rerank_rank_mismatches",
            "target_rank_mismatches", "ndcg_at_5_mismatches",
        ]
    elif family == "transformer":
        required = [
            "category_id", "pool_depth", "fallback_case_count",
            "candidate_id_or_order_mismatch_count",
            "feature_row_mismatch_count", "feature_cell_mismatch_count",
            "fold_mismatch_count", "logit_mismatch_count",
            "rerank_rank_mismatch_count", "target_rank_mismatch_count",
            "ndcg_at_5_mismatch_count", "mismatch_total", "identity_passed",
        ]
        depth_column = "pool_depth"
        fallback_column = "fallback_case_count"
        pass_column = "identity_passed"
        mismatch_columns = [
            "candidate_id_or_order_mismatch_count",
            "feature_row_mismatch_count", "feature_cell_mismatch_count",
            "fold_mismatch_count", "logit_mismatch_count",
            "rerank_rank_mismatch_count", "target_rank_mismatch_count",
            "ndcg_at_5_mismatch_count", "mismatch_total",
        ]
    else:
        raise ValueError(f"Unsupported finalized Full identity family: {family}")

    require_columns(qc, required, f"{family} Full fallback-identity QC")
    category_column = "category" if family == "lightgbm" else "category_id"
    observed_categories = set(qc[category_column].astype(str))
    allowed_categories = {CATEGORY_ID, CATEGORY_KEY, CATEGORY_LABEL}
    if not observed_categories or not observed_categories.issubset(allowed_categories):
        raise RuntimeError(
            f"{family} Full fallback-identity QC category mismatch: "
            f"observed={sorted(observed_categories)}, expected one of={sorted(allowed_categories)}"
        )
    observed_depths = sorted(
        pd.to_numeric(qc[depth_column], errors="raise").astype(int).tolist()
    )
    if observed_depths != CANDIDATE_POOL_DEPTHS:
        raise RuntimeError(
            f"{family} Full fallback-identity QC depth set differs from the canonical grid: "
            f"{observed_depths}"
        )

    fallback_counts = pd.to_numeric(
        qc[fallback_column], errors="raise"
    ).astype(int)
    if not fallback_counts.eq(int(expected_fallback_cases)).all():
        raise RuntimeError(
            f"{family} Full fallback count differs from Notebook 09: "
            f"expected={expected_fallback_cases}, observed={sorted(fallback_counts.unique())}"
        )

    mismatch_total = qc[mismatch_columns].apply(
        pd.to_numeric, errors="raise"
    ).to_numpy(dtype=np.int64).sum()
    passed = boolean_series(qc[pass_column]).fillna(False)
    if mismatch_total != 0 or not bool(passed.all()):
        raise RuntimeError(
            f"{family} Full fallback-identity QC failed: {path}"
        )

    return {
        "full_identity_qc_path": str(path),
        "full_identity_qc_sha256": file_sha256(path),
        "full_identity_qc_depths": ",".join(map(str, observed_depths)),
        "full_identity_qc_fallback_cases": int(expected_fallback_cases),
        "full_identity_qc_passed": True,
    }


def validate_finalized_source_readiness(source_spec, expected_fallback_cases):
    family = source_spec["method_family"]
    condition = source_spec["stage_condition"]
    run_manifest_path = Path(source_spec["run_manifest_path"])
    run_manifest = load_json_required(run_manifest_path, "reranker run manifest")

    if "category_id" not in run_manifest:
        raise RuntimeError(f"Reranker run manifest is missing category_id: {run_manifest_path}")
    if str(run_manifest["category_id"]) != CATEGORY_ID:
        raise RuntimeError(f"Reranker run manifest category mismatch: {run_manifest_path}")

    # Existing P2-Q/P2-P notebooks predate the downstream-ready fields.
    # When status fields exist they must be successful; Full must declare them.
    if "run_status" in run_manifest and run_manifest["run_status"] != "SUCCESS":
        raise RuntimeError(f"Upstream run is not SUCCESS: {run_manifest_path}")
    if "status" in run_manifest and str(run_manifest["status"]).lower() not in {
        "completed", "complete", "success"
    }:
        raise RuntimeError(f"Upstream run status is not completed: {run_manifest_path}")
    if "ready_for_downstream" in run_manifest and run_manifest["ready_for_downstream"] is not True:
        raise RuntimeError(f"Upstream run is not ready for downstream: {run_manifest_path}")

    if condition == "Full" and family in FULL_VALIDATION_REQUIRED_FAMILIES:
        if "run_status" in run_manifest and run_manifest.get("run_status") != "SUCCESS":
            raise RuntimeError(f"{family} Full must declare run_status=SUCCESS.")
        if "ready_for_downstream" in run_manifest and run_manifest.get("ready_for_downstream") is not True:
            raise RuntimeError(f"{family} Full must declare ready_for_downstream=true.")

    sha_result = {
        "model_sha_records_verified": 0,
        "model_contract_sha256_json": "[]",
    }
    if family in {"lightgbm", "transformer"}:
        sha_result = validate_model_artifact_sha_contracts(
            source_spec, f"{family} {condition}"
        )

    readiness = {
        "required_full_validation": bool(
            condition == "Full" and family in FULL_VALIDATION_REQUIRED_FAMILIES
        ),
        "full_validation_passed": True,
        "fallback_identity_passed": True,
        "zero_training_transfer_verified": True,
        "model_or_checkpoint_sha_consistent": True,
        "upstream_run_status": str(run_manifest.get("run_status", "")),
        "upstream_ready_for_downstream": run_manifest.get("ready_for_downstream"),
        **sha_result,
    }

    if condition == "Full" and family == "lightgbm":
        # Design_Lock rev4b self-pool: Full self-trains on S1-P; cold reuses Base (a=P2-Q) OOF (c-b01 cold == 0).
        require_manifest_value(
            run_manifest,
            ("shared_all_prior_model_role", "lightgbm_training_plan_role", "model_role"),
            {"train_and_export", "write"},
            "LightGBM Full run manifest",
        )
        require_manifest_value(
            run_manifest,
            ("candidate_pool_role", "candidate_source", "trained_on_pool"),
            {
                "personalized_retrieval",
                "personalized_winner",
                "personalized_winner_long",
                f"notebook09_personalized_winner_top1000_{CATEGORY_ID}",
            },
            "LightGBM Full run manifest",
        )
        require_manifest_value(
            run_manifest,
            (
                "method_family",
                "model_family",
                "reranker_family",
                "reranker_method",
                "rerank_method",
                "reranking_method",
            ),
            {"lightgbm", "lightgbm_rerank"},
            "LightGBM Full run manifest",
        )
        require_manifest_value(
            run_manifest,
            ("stage_condition", "condition_name", "condition", "experiment_condition"),
            {"full", "full_pipeline", "full_pipeline_personalization"},
            "LightGBM Full run manifest",
        )
        require_manifest_value(
            run_manifest,
            ("cold_metric_identity_passed",),
            {True},
            "LightGBM Full run manifest",
        )
        _lgbm_cold_qc = Path(str(
            run_manifest.get("cold_rerank_fallback_qc_path")
            or source_spec["run_dir"] / "cold_rerank_fallback_qc.csv"
        ))
        if _lgbm_cold_qc.exists():
            readiness["lightgbm_cold_identity_qc_path"] = str(_lgbm_cold_qc)

    elif condition == "Full" and family == "transformer":
        # Design_Lock rev4b self-pool (CONFIRMED vs 13c herbal run_manifest):
        # Full self-trains on S1-P; cold reuses Base (a=P2-Q) OOF -> cold_metric_identity_passed.
        if run_manifest.get("shared_all_prior_model_role") != "train_and_export":
            raise RuntimeError("Transformer Full must self-train (shared_all_prior_model_role != train_and_export).")
        require_true(
            run_manifest, "cold_metric_identity_passed",
            "Transformer Full run manifest",
        )
        require_manifest_value(
            run_manifest, ("checkpoint_selection_metric",), {"validation_ndcg_at_5"},
            "Transformer Full run manifest",
        )
        require_manifest_value(
            run_manifest, ("candidate_pool_role",), {"personalized_retrieval"},
            "Transformer Full run manifest",
        )
        require_manifest_value(
            run_manifest, ("cold_prediction_source",), {"p2q_cold_fallback"},
            "Transformer Full run manifest",
        )
        require_manifest_value(
            run_manifest,
            ("method_family", "model_family", "reranker_family", "reranker_method", "rerank_method"),
            {"transformer", "transformer_rerank"},
            "Transformer Full run manifest",
        )
        require_manifest_value(
            run_manifest,
            ("stage_condition", "condition_name", "condition", "experiment_condition"),
            {"full", "full_pipeline"},
            "Transformer Full run manifest",
        )
        _tf_cold_qc = Path(str(
            run_manifest.get("cold_rerank_fallback_qc_path")
            or source_spec["run_dir"] / "cold_rerank_fallback_qc.csv"
        ))
        if _tf_cold_qc.exists():
            readiness["transformer_cold_identity_qc_path"] = str(_tf_cold_qc)

    readiness["run_manifest_sha256"] = file_sha256(run_manifest_path)
    readiness["per_query_sha256"] = file_sha256(source_spec["per_query_path"])
    readiness["rank_artifact_sha256"] = file_sha256(source_spec["rank_path"])
    feature_manifest_path = Path(source_spec["feature_manifest_path"])
    readiness["feature_manifest_sha256"] = (
        file_sha256(feature_manifest_path) if feature_manifest_path.exists() else ""
    )
    interpretation_path = Path(source_spec["feature_interpretation_manifest_path"])
    readiness["interpretation_manifest_sha256"] = (
        file_sha256(interpretation_path) if interpretation_path.exists() else ""
    )
    return readiness



def validate_loaded_source_counts(
    source_spec,
    rank_rows,
    expected_case_ids,
):
    expected_case_ids = {str(value) for value in expected_case_ids}
    family = source_spec["method_family"]
    condition = source_spec["stage_condition"]

    count_rows = []
    for depth in CANDIDATE_POOL_DEPTHS:
        depth_rows = rank_rows.loc[
            pd.to_numeric(
                rank_rows["candidate_pool_depth"], errors="raise"
            ).eq(int(depth))
        ].copy()
        observed_case_ids = set(depth_rows["case_id"].astype(str))
        if observed_case_ids != expected_case_ids:
            raise RuntimeError(
                f"{family} {condition} case universe mismatch at depth={depth}: "
                f"missing={len(expected_case_ids - observed_case_ids)}, "
                f"extra={len(observed_case_ids - expected_case_ids)}"
            )
        if depth_rows.duplicated("case_id").any():
            raise RuntimeError(
                f"{family} {condition} has duplicate per-case rows at depth={depth}."
            )
        candidate_counts = pd.to_numeric(
            depth_rows["candidate_count"], errors="raise"
        ).astype(int)
        if not candidate_counts.eq(int(depth)).all():
            raise RuntimeError(
                f"{family} {condition} candidate counts are not exact-K at depth={depth}."
            )
        count_rows.append({
            "candidate_pool_depth": int(depth),
            "case_count": int(len(observed_case_ids)),
            "candidate_count_per_case": int(depth),
        })

    candidate_export_path = Path(source_spec["run_dir"]) / "reranked_candidates.parquet"
    candidate_export_validated = False
    candidate_export_sha256 = ""
    candidate_export_row_count = None

    if candidate_export_path.exists():
        try:
            import pyarrow.parquet as pq
            candidate_available_columns = set(
                pq.ParquetFile(candidate_export_path).schema.names
            )
        except ModuleNotFoundError:
            candidate_available_columns = set(
                pd.read_parquet(candidate_export_path).columns
            )
        candidate_id_column = next(
            (
                column
                for column in ["candidate_parent_asin", "candidate_item_id", "item_id", "parent_asin"]
                if column in candidate_available_columns
            ),
            None,
        )
        if candidate_id_column is None:
            raise RuntimeError(
                f"{family} {condition} candidate export has no candidate identifier."
            )
        candidate_required = [
            "case_id", "pool_depth", "rerank_method", candidate_id_column,
        ]
        candidate_df = read_parquet_projected(
            candidate_export_path, candidate_required
        ).rename(columns={candidate_id_column: "candidate_parent_asin"})
        candidate_df = candidate_df.loc[
            candidate_df["rerank_method"].astype(str).eq(
                source_spec["reranker_method"]
            )
        ].copy()
        if candidate_df.empty:
            raise RuntimeError(
                f"{family} {condition} candidate export has no rows for "
                f"{source_spec['reranker_method']}."
            )
        candidate_df["case_id"] = candidate_df["case_id"].astype(str)
        candidate_df["candidate_parent_asin"] = (
            candidate_df["candidate_parent_asin"].astype(str)
        )
        candidate_df["pool_depth"] = pd.to_numeric(
            candidate_df["pool_depth"], errors="raise"
        ).astype(int)
        if candidate_df.duplicated(
            ["case_id", "pool_depth", "candidate_parent_asin"]
        ).any():
            raise RuntimeError(
                f"{family} {condition} candidate export has duplicate candidate rows."
            )
        report_rows = candidate_df.loc[
            candidate_df["pool_depth"].eq(PRIMARY_REPORT_DEPTH)
        ].copy()
        report_case_ids = set(report_rows["case_id"])
        report_counts = report_rows.groupby("case_id").size()

        export_is_exact_report_depth = (
            report_case_ids == expected_case_ids
            and not report_counts.empty
            and bool(report_counts.eq(PRIMARY_REPORT_DEPTH).all())
        )

        if source_spec["rank_source"] == "reranked_candidates":
            if report_case_ids != expected_case_ids:
                raise RuntimeError(
                    f"{family} {condition} candidate export case universe is incomplete "
                    f"at depth={PRIMARY_REPORT_DEPTH}."
                )
            if not report_counts.eq(PRIMARY_REPORT_DEPTH).all():
                raise RuntimeError(
                    f"{family} {condition} candidate export is not exact-{PRIMARY_REPORT_DEPTH}."
                )

        candidate_export_validated = bool(export_is_exact_report_depth)
        candidate_export_sha256 = file_sha256(candidate_export_path)
        candidate_export_row_count = int(len(candidate_df))
    elif family in {"lightgbm", "transformer"}:
        raise RuntimeError(
            f"{family} {condition} is missing its candidate-level export: '{candidate_export_path}'"
        )

    return {
        "case_count_consistent": True,
        "candidate_count_consistent": True,
        "count_contract_json": json.dumps(
            count_rows, ensure_ascii=False, sort_keys=True
        ),
        "candidate_export_path": (
            str(candidate_export_path) if candidate_export_path.exists() else ""
        ),
        "candidate_export_validated": bool(candidate_export_validated),
        "candidate_export_sha256": candidate_export_sha256,
        "candidate_export_row_count": candidate_export_row_count,
    }

In [6]:
# ==== SELF-CHECK SC-3: Notebook 14 Canonical Readiness Invariants ====
def _sc3_require_final_full_manifest_fields(run_manifest, expected_category, expected_family, expected_condition):
    missing = [
        key for key in ["run_status", "ready_for_downstream", "category_id"]
        if key not in run_manifest
    ]
    if missing:
        raise RuntimeError(f"SC-3 Full manifest is missing required fields: {missing}")
    if run_manifest["run_status"] != "SUCCESS":
        raise RuntimeError("SC-3 Full manifest is not SUCCESS.")
    if run_manifest["ready_for_downstream"] is not True:
        raise RuntimeError("SC-3 Full manifest is not ready for downstream.")
    if str(run_manifest["category_id"]) != str(expected_category):
        raise RuntimeError("SC-3 Full manifest category mismatch.")
    family_values = {
        str(run_manifest.get(key, "")).strip().lower()
        for key in ("method_family", "model_family", "reranker_family", "reranker_method", "rerank_method")
        if run_manifest.get(key) is not None
    }
    if str(expected_family).lower() not in family_values:
        raise RuntimeError("SC-3 Full manifest method-family mismatch.")
    condition_values = {
        str(run_manifest.get(key, "")).strip().lower()
        for key in ("stage_condition", "condition_name", "condition", "experiment_condition")
        if run_manifest.get(key) is not None
    }
    if str(expected_condition).lower() not in condition_values and "full_pipeline" not in condition_values:
        raise RuntimeError("SC-3 Full manifest stage-condition mismatch.")
    return True


def _sc3_canonical_readiness_dependency(primary_contract_present, interpretation_manifest_present):
    if primary_contract_present is not True:
        raise RuntimeError("SC-3 primary model/checkpoint contract is required.")
    return {
        "primary_contract_present": True,
        "interpretation_manifest_present": bool(interpretation_manifest_present),
        "canonical_readiness_passed": True,
    }


def _sc3_expect_raises(label, fn):
    try:
        fn()
    except Exception:
        return {"test": label, "raised": True}
    raise RuntimeError(f"SC-3 negative test did not raise: {label}")


_sc3_require_final_full_manifest_fields(
    {
        "run_status": "SUCCESS",
        "ready_for_downstream": True,
        "category_id": CATEGORY_ID,
        "method_family": "lightgbm",
        "stage_condition": "Full",
    },
    CATEGORY_ID,
    "lightgbm",
    "Full",
)
_sc3_canonical_readiness_dependency(
    primary_contract_present=True,
    interpretation_manifest_present=False,
)
_sc3_nb14_negative_results = [
    _sc3_expect_raises(
        "interpretation manifest exists but primary contract missing",
        lambda: _sc3_canonical_readiness_dependency(False, True),
    ),
    _sc3_expect_raises(
        "missing run_status",
        lambda: _sc3_require_final_full_manifest_fields(
            {"ready_for_downstream": True, "category_id": CATEGORY_ID, "method_family": "lightgbm", "stage_condition": "Full"},
            CATEGORY_ID, "lightgbm", "Full",
        ),
    ),
    _sc3_expect_raises(
        "wrong method family",
        lambda: _sc3_require_final_full_manifest_fields(
            {"run_status": "SUCCESS", "ready_for_downstream": True, "category_id": CATEGORY_ID, "method_family": "gam", "stage_condition": "Full"},
            CATEGORY_ID, "lightgbm", "Full",
        ),
    ),
]
SC3_NOTEBOOK14_SELF_CHECK_PASSED = True
sc3_notebook14_self_check_results = pd.DataFrame(_sc3_nb14_negative_results)
display(sc3_notebook14_self_check_results)
print("SC-3 Notebook 14 readiness synthetic checks passed.")


,test,raised
0,interpretation manifest exists but primary con...,True
1,missing run_status,True
2,wrong method family,True


SC-3 Notebook 14 readiness synthetic checks passed.


In [7]:
# ==== Runtime Memory Cleanup Before Large Artifact Loads ====
import gc

_reclaimed_names = []
for _name, _value in list(globals().items()):
    if _name.startswith("_"):
        continue
    if isinstance(_value, (pd.DataFrame, pd.Series)):
        _reclaimed_names.append(_name)
        globals().pop(_name, None)
    elif isinstance(_value, np.ndarray) and _value.nbytes >= 5_000_000:
        _reclaimed_names.append(_name)
        globals().pop(_name, None)

if "torch" in globals():
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

_reclaimed_count = gc.collect()
print(
    "Pre-load memory cleanup complete: "
    f"dropped {len(_reclaimed_names)} large globals; "
    f"gc reclaimed {_reclaimed_count} objects."
)
if _reclaimed_names:
    print("Dropped globals:", sorted(_reclaimed_names))


Pre-load memory cleanup complete: dropped 1 large globals; gc reclaimed 31 objects.
Dropped globals: ['sc3_notebook14_self_check_results']


In [8]:
# ==== Authoritative Notebook 09 Pools and Reranker per-case Ranks ====
if not STAGE1_MANIFEST_PATH.exists():
    raise RuntimeError(f"Missing Notebook 09 manifest: {STAGE1_MANIFEST_PATH}")

stage1_manifest = json.loads(STAGE1_MANIFEST_PATH.read_text(encoding="utf-8"))
required_manifest_keys = [
    "notebook_name", "output_paths", "candidate_budget_k",
    "complete_case_universe_preserved",
    "fallback_case_count",
]
missing_manifest_keys = sorted(set(required_manifest_keys).difference(stage1_manifest))
if missing_manifest_keys:
    raise RuntimeError(
        f"Notebook 09 manifest is missing required keys: {missing_manifest_keys}"
    )
if int(stage1_manifest["candidate_budget_k"]) != max(CANDIDATE_POOL_DEPTHS):
    raise RuntimeError("Notebook 09 candidate budget does not match the canonical depth contract.")
if not bool(stage1_manifest["complete_case_universe_preserved"]):
    raise RuntimeError("Notebook 09 did not preserve the complete case universe.")
if "qchs_fallback_cases_preserved" in stage1_manifest:
    if not bool(stage1_manifest["qchs_fallback_cases_preserved"]):
        raise RuntimeError("Notebook 09 did not preserve QCHS fallback cases.")
else:
    print("Notebook 09 manifest lacks qchs_fallback_cases_preserved; using downstream fallback-count validation.")

if int(stage1_manifest["fallback_case_count"]) != FROZEN_FALLBACK_CASE_COUNT:
    raise RuntimeError(
        "Notebook 09 fallback count differs from the frozen benchmark contract: "
        f"expected={FROZEN_FALLBACK_CASE_COUNT}, "
        f"observed={stage1_manifest['fallback_case_count']}"
    )

stage1_output_paths = stage1_manifest["output_paths"]
for required_key in ["query_only_winner_long", "personalized_winner_long"]:
    if required_key not in stage1_output_paths:
        raise RuntimeError(f"Notebook 09 manifest is missing output path: {required_key}")

stage1_input_paths = stage1_manifest.get("input_paths", {})
for required_key in ["notebook07_winner_contract", "notebook08_winner_contract"]:
    if required_key not in stage1_input_paths:
        raise RuntimeError(f"Notebook 09 manifest is missing winner-lineage path: {required_key}")
notebook07_winner_manifest_path = Path(stage1_input_paths["notebook07_winner_contract"])
notebook08_winner_manifest_path = Path(stage1_input_paths["notebook08_winner_contract"])
if not notebook07_winner_manifest_path.exists() or not notebook08_winner_manifest_path.exists():
    raise RuntimeError(
        "Notebook 07/08 winner manifests referenced by Notebook 09 are missing: "
        f"{notebook07_winner_manifest_path}, {notebook08_winner_manifest_path}"
    )
notebook07_winner_manifest = json.loads(
    notebook07_winner_manifest_path.read_text(encoding="utf-8")
)
notebook08_winner_manifest = json.loads(
    notebook08_winner_manifest_path.read_text(encoding="utf-8")
)
if notebook07_winner_manifest.get("winner_method_key") != stage1_manifest.get(
    "baseline_retrieval_winner_method_key"
):
    raise RuntimeError("Notebook 07 winner contract disagrees with Notebook 09 P0 lineage.")
if notebook08_winner_manifest.get("winner_method_slug") != stage1_manifest.get(
    "selected_personalized_method_slug"
):
    raise RuntimeError("Notebook 08 winner contract disagrees with Notebook 09 P1 lineage.")

p0_pool_path = Path(stage1_output_paths["query_only_winner_long"])
p1_pool_path = Path(stage1_output_paths["personalized_winner_long"])
if not p0_pool_path.exists() or not p1_pool_path.exists():
    raise RuntimeError(
        f"Notebook 09 authoritative pools are missing: P0={p0_pool_path}, P1={p1_pool_path}"
    )

stage1_required_pool_columns = [
    "case_id", "query_id", "user_id", "regime", "target_parent_asin",
    "candidate_parent_asin", "candidate_rank", "retrieval_method", "is_gt",
]
stage1_optional_pool_columns = [
    "qchs_profile_available", "profile_fallback_flag", "profile_fallback_reason",
]
p0_pool_df = read_parquet_projected(
    p0_pool_path, stage1_required_pool_columns, stage1_optional_pool_columns
)
p1_pool_df = read_parquet_projected(
    p1_pool_path, stage1_required_pool_columns, stage1_optional_pool_columns
)
p1_fallback_column_available = "profile_fallback_flag" in p1_pool_df.columns
p1_qchs_column_available = "qchs_profile_available" in p1_pool_df.columns

p0_rank_rows_df = build_stage1_rank_rows(
    p0_pool_df, "P0", "query_only_winner_long",
    stage1_manifest["notebook_name"], p0_pool_path, False,
)
p1_rank_rows_df = build_stage1_rank_rows(
    p1_pool_df, "P1-only", "personalized_winner_long",
    stage1_manifest["notebook_name"], p1_pool_path, True,
)

p0_case_meta_df = p0_rank_rows_df.loc[
    p0_rank_rows_df["candidate_pool_depth"].eq(max(CANDIDATE_POOL_DEPTHS))
].drop_duplicates("case_id")
p1_case_meta_df = p1_rank_rows_df.loc[
    p1_rank_rows_df["candidate_pool_depth"].eq(max(CANDIDATE_POOL_DEPTHS))
].drop_duplicates("case_id")

source_inventory_rows = [
    {
        "source_notebook": stage1_manifest["notebook_name"],
        "source_path": str(p0_pool_path),
        "metric_validation_path": "",
        "condition": "P0",
        "reranker": "none",
        "method_family": "shared_retrieval_baseline",
        "record_type": "authoritative_stage1_original_order",
        "available_candidate_pool_depths": ",".join(map(str, CANDIDATE_POOL_DEPTHS)),
        "available_metric_cutoffs": ",".join(map(str, METRIC_CUTOFFS)),
        "row_count": int(len(p0_pool_df)),
        "canonical_rank_row_count": int(len(p0_rank_rows_df)),
        "case_count": int(p0_rank_rows_df["case_id"].nunique()),
        "regime_counts": regime_counts_json(p0_rank_rows_df),
        "load_status": "loaded",
    },
    {
        "source_notebook": stage1_manifest["notebook_name"],
        "source_path": str(p1_pool_path),
        "metric_validation_path": "",
        "condition": "P1-only",
        "reranker": "none",
        "method_family": "shared_retrieval_baseline",
        "record_type": "authoritative_stage1_original_order",
        "available_candidate_pool_depths": ",".join(map(str, CANDIDATE_POOL_DEPTHS)),
        "available_metric_cutoffs": ",".join(map(str, METRIC_CUTOFFS)),
        "row_count": int(len(p1_pool_df)),
        "canonical_rank_row_count": int(len(p1_rank_rows_df)),
        "case_count": int(p1_rank_rows_df["case_id"].nunique()),
        "regime_counts": regime_counts_json(p1_rank_rows_df),
        "load_status": "loaded",
    },
]

stage2_rank_frames = []
source_metric_disagreements = []
source_contract_gate_rows = []
expected_fallback_cases = int(stage1_manifest["fallback_case_count"])
for source_spec in SOURCE_RUNS:
    uses_personalized_pool = source_spec["stage_condition"] in {"Full", "b02"}
    authoritative_meta = p1_case_meta_df if uses_personalized_pool else p0_case_meta_df
    expected_candidate_pool_path = p1_pool_path if uses_personalized_pool else p0_pool_path
    base_gate = validate_source_contract(
        source_spec, expected_candidate_pool_path
    )
    readiness_gate = validate_finalized_source_readiness(
        source_spec, expected_fallback_cases
    )
    rank_rows, inventory_row, disagreements = load_stage2_rank_rows(
        source_spec, authoritative_meta
    )
    expected_case_ids = set(authoritative_meta["case_id"].astype(str))
    count_gate = validate_loaded_source_counts(
        source_spec, rank_rows, expected_case_ids
    )
    source_contract_gate_rows.append({
        **base_gate,
        **readiness_gate,
        **count_gate,
        "gate_status": "SUCCESS",
    })
    stage2_rank_frames.append(rank_rows)
    source_inventory_rows.append(inventory_row)
    source_metric_disagreements.extend(disagreements)

source_metric_disagreement_df = pd.DataFrame(source_metric_disagreements)
if source_metric_disagreements:
    print(
        "WARNING: upstream metric columns disagree with canonical one-based "
        "target-rank reconstruction. Notebook 14 retains the rank-derived "
        f"canonical values and records {len(source_metric_disagreements)} disagreements."
    )
else:
    source_metric_disagreement_df = pd.DataFrame(
        columns=[
            "source_notebook", "condition", "reranker", "case_id",
            "candidate_pool_depth", "target_rank", "source_metric",
            "reconstructed_metric", "source_metric_name",
        ]
    )

pipeline_source_inventory_df = pd.DataFrame(source_inventory_rows)
pipeline_source_contract_gate_df = pd.DataFrame(source_contract_gate_rows)
if not pipeline_source_contract_gate_df["gate_status"].eq("SUCCESS").all():
    raise RuntimeError("At least one reranker source contract gate failed.")

required_readiness_columns = [
    "full_validation_passed", "fallback_identity_passed",
    "zero_training_transfer_verified", "model_or_checkpoint_sha_consistent",
    "case_count_consistent", "candidate_count_consistent",
]
for readiness_column in required_readiness_columns:
    if not boolean_series(
        pipeline_source_contract_gate_df[readiness_column]
    ).fillna(False).all():
        raise RuntimeError(
            f"At least one upstream source failed readiness check: {readiness_column}"
        )

pipeline_upstream_readiness_df = pipeline_source_contract_gate_df.copy()
source_hash_columns = [
    "category_id", "stage_condition", "method_family", "reranker_method",
    "run_manifest_path", "run_manifest_sha256", "per_query_sha256",
    "rank_artifact_sha256", "feature_manifest_sha256",
    "full_identity_qc_path", "full_identity_qc_sha256",
    "candidate_export_path", "candidate_export_sha256",
    "model_contract_sha256_json",
]
pipeline_source_hashes_df = pipeline_source_contract_gate_df[
    [column for column in source_hash_columns
     if column in pipeline_source_contract_gate_df.columns]
].copy()

# Stage-1 pools and winner-lineage manifests are part of the canonical source chain.
stage1_manifest_sha256 = file_sha256(STAGE1_MANIFEST_PATH)
notebook07_winner_sha256 = file_sha256(notebook07_winner_manifest_path)
notebook08_winner_sha256 = file_sha256(notebook08_winner_manifest_path)
stage1_hash_rows = []
for stage_condition, pool_path, winner_manifest_path, winner_sha256 in [
    ("P0", p0_pool_path, notebook07_winner_manifest_path, notebook07_winner_sha256),
    ("P1-only", p1_pool_path, notebook08_winner_manifest_path, notebook08_winner_sha256),
]:
    pool_sha256 = file_sha256(pool_path)
    stage1_hash_rows.append({
        "category_id": CATEGORY_ID,
        "stage_condition": stage_condition,
        "method_family": "shared_retrieval_baseline",
        "reranker_method": "none",
        "run_manifest_path": str(STAGE1_MANIFEST_PATH),
        "run_manifest_sha256": stage1_manifest_sha256,
        "per_query_sha256": pool_sha256,
        "rank_artifact_sha256": pool_sha256,
        "feature_manifest_sha256": winner_sha256,
        "full_identity_qc_path": "",
        "full_identity_qc_sha256": "",
        "candidate_export_path": str(pool_path),
        "candidate_export_sha256": pool_sha256,
        "model_contract_sha256_json": json.dumps(
            [file_sha256(winner_manifest_path)], ensure_ascii=False
        ),
    })
pipeline_source_hashes_df = pd.concat(
    [pd.DataFrame(stage1_hash_rows), pipeline_source_hashes_df],
    ignore_index=True,
    sort=False,
)

for method_family in ["shannon", "lightgbm", "gam", "transformer"]:
    pair = pipeline_source_contract_gate_df.loc[
        pipeline_source_contract_gate_df["method_family"].eq(method_family)
        & pipeline_source_contract_gate_df["stage_condition"].isin(["P2-P", "Full"])
    ].copy()
    if len(pair) != 2:
        raise RuntimeError(f"Missing P2-P/Full source-contract pair for {method_family}.")
    if pair["primary_feature_contract_hash"].nunique(dropna=False) != 1:
        raise RuntimeError(f"P2-P/Full primary feature contract differs for {method_family}.")
    if method_family == "shannon" and pair["normalization_contract_hash"].nunique(dropna=False) != 1:
        raise RuntimeError("Shannon P2-P/Full normalization contract differs.")

canonical_rank_rows_df = pd.concat(
    [p0_rank_rows_df, p1_rank_rows_df, *stage2_rank_frames],
    ignore_index=True,
    sort=False,
)

del p0_pool_df, p1_pool_df, p0_rank_rows_df, p1_rank_rows_df, p0_case_meta_df, stage2_rank_frames
gc.collect()


Notebook 09 manifest lacks qchs_fallback_cases_preserved; using downstream fallback-count validation.


150

In [9]:
# ==== Canonical Long per-case metrics; No Aggregation Before This Point ====
raw_frames = []
provenance_columns = [
    "case_id", "query_id", "user_id", "regime", "stage_condition",
    "reranker_method", "method_family", "candidate_pool_depth",
    "target_parent_asin", "target_exposed", "target_rank", "candidate_count",
    "candidate_source", "retrieval_method", "personalization_active",
    "qchs_profile_available", "profile_fallback_flag", "profile_fallback_reason",
    "source_notebook", "source_path", "record_type", "shared_stage1_baseline",
]
require_columns(canonical_rank_rows_df, provenance_columns, "canonical rank rows")

for metric_name, metric_cutoff in itertools.product(METRIC_NAMES, METRIC_CUTOFFS):
    valid = canonical_rank_rows_df["candidate_pool_depth"].ge(int(metric_cutoff))
    if not valid.any():
        continue
    metric_rows = canonical_rank_rows_df.loc[valid, provenance_columns].copy()
    metric_rows["metric_name"] = metric_name
    metric_rows["metric_cutoff"] = int(metric_cutoff)
    metric_rows["metric_value"] = metric_from_rank(
        metric_rows["target_rank"], metric_name, metric_cutoff
    )
    metric_rows["shared_baseline_repeated_for_display"] = False
    raw_frames.append(metric_rows)

pipeline_canonical_per_case_metrics_df = pd.concat(
    raw_frames, ignore_index=True, sort=False
)
canonical_column_order = [
    "case_id", "query_id", "user_id", "regime", "stage_condition",
    "reranker_method", "method_family", "candidate_pool_depth",
    "metric_name", "metric_cutoff", "metric_value",
    "target_parent_asin", "target_exposed", "target_rank", "candidate_count",
    "candidate_source", "retrieval_method", "personalization_active",
    "qchs_profile_available", "profile_fallback_flag", "profile_fallback_reason",
    "source_notebook", "source_path", "record_type", "shared_stage1_baseline",
    "shared_baseline_repeated_for_display",
]
pipeline_canonical_per_case_metrics_df = pipeline_canonical_per_case_metrics_df[
    canonical_column_order
].sort_values(
    [
        "stage_condition", "method_family", "reranker_method", "case_id",
        "candidate_pool_depth", "metric_name", "metric_cutoff",
    ],
    kind="mergesort",
).reset_index(drop=True)

canonical_key = [
    "case_id", "stage_condition", "reranker_method",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
]
duplicate_key_rows = pipeline_canonical_per_case_metrics_df.loc[
    pipeline_canonical_per_case_metrics_df.duplicated(canonical_key, keep=False)
]
if not duplicate_key_rows.empty:
    raise RuntimeError(
        "Duplicated canonical key rows: "
        + duplicate_key_rows[canonical_key].head(10).to_json(orient="records")
    )

del raw_frames, duplicate_key_rows
gc.collect()


# Locked headline authority: complete case × condition × reranker-family grid.
_primary_normalized = pipeline_canonical_per_case_metrics_df.loc[
    pipeline_canonical_per_case_metrics_df["candidate_pool_depth"].eq(REPORT_POOL_DEPTH)
    & pipeline_canonical_per_case_metrics_df["metric_name"].eq(PRIMARY_METRIC_NAME)
    & pipeline_canonical_per_case_metrics_df["metric_cutoff"].eq(PRIMARY_METRIC_CUTOFF)
].copy()
if _primary_normalized.empty:
    raise RuntimeError("The canonical depth-1000 unconditional NDCG@5 slice is empty.")

_primary_shared = _primary_normalized.loc[
    _primary_normalized["stage_condition"].isin(["P0", "P1-only"])
].copy()
_primary_stage2 = _primary_normalized.loc[
    _primary_normalized["stage_condition"].isin(["P2-Q", "P2-P", "b02", "Full"])
].copy()

_primary_frames = []
for _family in RERANKER_FAMILIES:
    _shared_family = _primary_shared.copy()
    _shared_family["analysis_method_family"] = _family
    _shared_family["shared_baseline_repeated_for_method_grid"] = True
    _primary_frames.append(_shared_family)

_primary_stage2["analysis_method_family"] = _primary_stage2["method_family"].astype(str)
_primary_stage2["shared_baseline_repeated_for_method_grid"] = False
_primary_frames.append(_primary_stage2)

pipeline_canonical_primary_ndcg5_depth1000_df = pd.concat(
    _primary_frames, ignore_index=True, sort=False
).sort_values(
    ["analysis_method_family", "stage_condition", "case_id"],
    kind="mergesort",
).reset_index(drop=True)

_primary_key = [
    "case_id", "stage_condition", "analysis_method_family",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
]
if pipeline_canonical_primary_ndcg5_depth1000_df.duplicated(_primary_key).any():
    raise RuntimeError("The locked primary method grid contains duplicate keys.")

_primary_case_ids = set(_primary_normalized["case_id"].astype(str))
PRIMARY_EXPECTED_METHOD_CONDITION_PAIRS = {
    (family, condition)
    for family in RERANKER_FAMILIES
    for condition in ["P0", "P1-only"]
}.union(
    {
        (source_spec["method_family"], source_spec["stage_condition"])
        for source_spec in SOURCE_RUNS
    }
)
_observed_primary_pairs = set(
    pipeline_canonical_primary_ndcg5_depth1000_df[
        ["analysis_method_family", "stage_condition"]
    ].drop_duplicates().itertuples(index=False, name=None)
)
if _observed_primary_pairs != PRIMARY_EXPECTED_METHOD_CONDITION_PAIRS:
    raise RuntimeError(
        "The locked primary method grid has unexpected method/condition pairs: "
        f"missing={sorted(PRIMARY_EXPECTED_METHOD_CONDITION_PAIRS - _observed_primary_pairs)}, "
        f"extra={sorted(_observed_primary_pairs - PRIMARY_EXPECTED_METHOD_CONDITION_PAIRS)}"
    )

_expected_primary_rows = (
    len(_primary_case_ids)
    * len(PRIMARY_EXPECTED_METHOD_CONDITION_PAIRS)
)
if len(pipeline_canonical_primary_ndcg5_depth1000_df) != _expected_primary_rows:
    raise RuntimeError(
        "The locked primary method grid is incomplete: "
        f"expected={_expected_primary_rows}, "
        f"observed={len(pipeline_canonical_primary_ndcg5_depth1000_df)}"
    )
for (_family, _condition), _cell in (
    pipeline_canonical_primary_ndcg5_depth1000_df
    .groupby(["analysis_method_family", "stage_condition"], observed=True)
):
    if (_family, _condition) not in PRIMARY_EXPECTED_METHOD_CONDITION_PAIRS:
        raise RuntimeError(
            f"Unexpected primary method grid pair: {_family} / {_condition}"
        )
    if set(_cell["case_id"].astype(str)) != _primary_case_ids:
        raise RuntimeError(
            f"Incomplete primary case grid: {_family} / {_condition}"
        )

del (
    _primary_normalized, _primary_shared, _primary_stage2,
    _primary_frames, _shared_family,
)
gc.collect()


0

In [10]:
# ==== Detailed Summaries and five-condition Display View ====
regime_group = [
    "reranker_method", "method_family", "stage_condition", "regime",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
]
pipeline_results_by_regime_pool_depth_df = (
    pipeline_canonical_per_case_metrics_df
    .groupby(regime_group, dropna=False, observed=True)
    .agg(
        case_count=("case_id", "nunique"),
        metric_mean=("metric_value", "mean"),
        metric_std=("metric_value", "std"),
        metric_min=("metric_value", "min"),
        metric_median=("metric_value", "median"),
        metric_max=("metric_value", "max"),
        target_exposure_count=("target_exposed", "sum"),
        target_exposure_rate=("target_exposed", "mean"),
    )
    .reset_index()
)
pipeline_results_by_regime_pool_depth_df["case_count"] = (
    pipeline_results_by_regime_pool_depth_df["case_count"].astype(int)
)
pipeline_results_by_regime_pool_depth_df["target_exposure_count"] = (
    pipeline_results_by_regime_pool_depth_df["target_exposure_count"].astype(int)
)

regime_index = [
    "reranker_method", "method_family", "stage_condition",
    "regime", "metric_name", "metric_cutoff",
]
regime_mean_wide = pipeline_results_by_regime_pool_depth_df.pivot(
    index=regime_index,
    columns="candidate_pool_depth",
    values="metric_mean",
).rename(columns=lambda depth: f"metric_mean_depth_{int(depth)}")
regime_count_wide = pipeline_results_by_regime_pool_depth_df.pivot(
    index=regime_index,
    columns="candidate_pool_depth",
    values="case_count",
).rename(columns=lambda depth: f"case_count_depth_{int(depth)}")
pipeline_results_by_regime_df = (
    regime_mean_wide.join(regime_count_wide, how="outer").reset_index()
)
pipeline_results_by_regime_df.columns.name = None

pool_group = [
    "reranker_method", "method_family", "stage_condition",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
]
micro_df = (
    pipeline_canonical_per_case_metrics_df
    .groupby(pool_group, dropna=False, observed=True)
    .agg(
        case_count=("case_id", "nunique"),
        micro_metric_mean=("metric_value", "mean"),
        target_exposure_rate=("target_exposed", "mean"),
    )
    .reset_index()
)
regime_means_df = (
    pipeline_canonical_per_case_metrics_df
    .groupby([*pool_group, "regime"], dropna=False, observed=True)["metric_value"]
    .mean()
    .reset_index(name="regime_metric_mean")
)
macro_df = (
    regime_means_df
    .groupby(pool_group, dropna=False, observed=True)
    .agg(
        regime_count=("regime", "nunique"),
        regime_macro_metric_mean=("regime_metric_mean", "mean"),
    )
    .reset_index()
)
pipeline_results_by_pool_depth_df = micro_df.merge(
    macro_df, on=pool_group, how="left", validate="one_to_one"
)

pipeline_results_overall_df = (
    pipeline_canonical_per_case_metrics_df
    .groupby(pool_group, dropna=False, observed=True)
    .agg(
        case_count=("case_id", "nunique"),
        regime_count=("regime", "nunique"),
        metric_mean=("metric_value", "mean"),
        metric_std=("metric_value", "std"),
        metric_min=("metric_value", "min"),
        metric_median=("metric_value", "median"),
        metric_max=("metric_value", "max"),
        target_exposure_count=("target_exposed", "sum"),
        target_exposure_rate=("target_exposed", "mean"),
    )
    .reset_index()
)

stage2_display_df = pipeline_results_by_regime_pool_depth_df.loc[
    pipeline_results_by_regime_pool_depth_df["stage_condition"].isin(
        ["P2-Q", "P2-P", "Full"]
    )
].copy()
stage2_display_df["display_reranker_family"] = stage2_display_df["method_family"]
stage2_display_df["shared_baseline_repeated_for_display"] = False

shared_display_source_df = pipeline_results_by_regime_pool_depth_df.loc[
    pipeline_results_by_regime_pool_depth_df["stage_condition"].isin(["P0", "P1-only"])
].copy()
shared_display_frames = []
for reranker_family in RERANKER_FAMILIES:
    repeated = shared_display_source_df.copy()
    repeated["display_reranker_family"] = reranker_family
    repeated["shared_baseline_repeated_for_display"] = True
    shared_display_frames.append(repeated)

pipeline_five_condition_comparison_df = pd.concat(
    [*shared_display_frames, stage2_display_df],
    ignore_index=True,
    sort=False,
)
pipeline_five_condition_comparison_df["stage_condition"] = pd.Categorical(
    pipeline_five_condition_comparison_df["stage_condition"],
    categories=CANONICAL_CONDITIONS,
    ordered=True,
)
pipeline_five_condition_comparison_df = pipeline_five_condition_comparison_df.sort_values(
    [
        "display_reranker_family", "metric_name", "metric_cutoff", "regime",
        "candidate_pool_depth", "stage_condition",
    ],
    kind="mergesort",
).reset_index(drop=True)
pipeline_five_condition_comparison_df["stage_condition"] = (
    pipeline_five_condition_comparison_df["stage_condition"].astype(str)
)

del (
    regime_mean_wide, regime_count_wide, micro_df, regime_means_df, macro_df,
    stage2_display_df, shared_display_source_df, shared_display_frames, repeated,
)
gc.collect()


0

In [11]:
# ==== Full method/depth/cutoff/regime Availability Inventory ====
regimes = sorted(
    pipeline_canonical_per_case_metrics_df["regime"].dropna().astype(str).unique().tolist()
)
# ---------------------------------------------------------
# PATCH 2026-07-26 (design registry). RERANKER_FAMILIES x CANONICAL_CONDITIONS is a
# coordinate grid, not the experimental design. The 2x2 stage allocation instantiates
# b02 (RetrP) only for families that can be retrained on the S1-P pool; Shannon is
# training-free, no 10b_02 notebook exists, and (shannon, b02) is therefore absent by
# construction. Enumerating the bare cartesian product manufactured phantom cells for
# that pair and failed `no_missing_expected_cells`. The registry below derives the
# expectation from SOURCE_RUNS so that absence-by-design and absence-by-fault remain
# distinguishable rather than being collapsed into one status.
# ---------------------------------------------------------
SHARED_STAGE1_CONDITIONS = ["P0", "P1-only"]
DESIGN_METHOD_CONDITION_PAIRS = {
    (family, condition)
    for family in RERANKER_FAMILIES
    for condition in SHARED_STAGE1_CONDITIONS
}.union(
    {
        (source_spec["method_family"], source_spec["stage_condition"])
        for source_spec in SOURCE_RUNS
    }
)
DESIGN_EXCLUDED_METHOD_CONDITION_PAIRS = sorted(
    {
        (family, condition)
        for family in RERANKER_FAMILIES
        for condition in CANONICAL_CONDITIONS
    }.difference(DESIGN_METHOD_CONDITION_PAIRS)
)

availability_rows = []
for reranker_family, condition, depth, metric_name, cutoff, regime in itertools.product(
    RERANKER_FAMILIES,
    CANONICAL_CONDITIONS,
    CANDIDATE_POOL_DEPTHS,
    METRIC_NAMES,
    METRIC_CUTOFFS,
    regimes,
):
    shared_display = condition in {"P0", "P1-only"}
    if shared_display:
        cell = pipeline_canonical_per_case_metrics_df.loc[
            pipeline_canonical_per_case_metrics_df["stage_condition"].eq(condition)
            & pipeline_canonical_per_case_metrics_df["reranker_method"].eq("none")
            & pipeline_canonical_per_case_metrics_df["candidate_pool_depth"].eq(depth)
            & pipeline_canonical_per_case_metrics_df["metric_name"].eq(metric_name)
            & pipeline_canonical_per_case_metrics_df["metric_cutoff"].eq(cutoff)
            & pipeline_canonical_per_case_metrics_df["regime"].astype(str).eq(regime)
        ]
    else:
        cell = pipeline_canonical_per_case_metrics_df.loc[
            pipeline_canonical_per_case_metrics_df["stage_condition"].eq(condition)
            & pipeline_canonical_per_case_metrics_df["method_family"].eq(reranker_family)
            & pipeline_canonical_per_case_metrics_df["candidate_pool_depth"].eq(depth)
            & pipeline_canonical_per_case_metrics_df["metric_name"].eq(metric_name)
            & pipeline_canonical_per_case_metrics_df["metric_cutoff"].eq(cutoff)
            & pipeline_canonical_per_case_metrics_df["regime"].astype(str).eq(regime)
        ]

    in_design = (reranker_family, condition) in DESIGN_METHOD_CONDITION_PAIRS
    if not in_design:
        availability_status = "not_run_by_design"
    elif cutoff > depth:
        availability_status = "invalid_cutoff_for_pool"
    elif not cell.empty:
        availability_status = "available"
    else:
        availability_status = "missing_unexpected"

    source_reranker_methods = (
        ",".join(sorted(cell["reranker_method"].astype(str).unique()))
        if not cell.empty else ""
    )
    availability_rows.append({
        "reranker_family": reranker_family,
        "stage_condition": condition,
        "candidate_pool_depth": int(depth),
        "metric_name": metric_name,
        "metric_cutoff": int(cutoff),
        "regime": regime,
        "availability_status": availability_status,
        "in_design": bool(in_design),
        "case_count": int(cell["case_id"].nunique()) if not cell.empty else 0,
        "source_reranker_method": source_reranker_methods,
        "shared_baseline_repeated_for_display": bool(shared_display),
    })

pipeline_method_depth_availability_df = pd.DataFrame(availability_rows)

# PATCH 2026-07-26: emit the inventory breakdown here, upstream of the cell-13 raise,
# so any residual gap is diagnosable from a single run instead of by bisection.
_availability_status_counts = (
    pipeline_method_depth_availability_df["availability_status"].value_counts().to_dict()
)
print("Availability inventory:", _availability_status_counts)
print("Design-excluded (family, condition) pairs:", DESIGN_EXCLUDED_METHOD_CONDITION_PAIRS)
_unexpected_cells_df = pipeline_method_depth_availability_df.loc[
    pipeline_method_depth_availability_df["availability_status"].eq("missing_unexpected")
]
if not _unexpected_cells_df.empty:
    print("missing_unexpected breakdown:")
    print(
        _unexpected_cells_df
        .groupby(
            ["reranker_family", "stage_condition", "candidate_pool_depth", "regime"],
            observed=True,
        )
        .size().rename("cells").reset_index().to_string(index=False)
    )


Availability inventory: {'available': 6210, 'invalid_cutoff_for_pool': 2070, 'not_run_by_design': 360}
Design-excluded (family, condition) pairs: [('shannon', 'b02')]


In [12]:
# ==== Contract validation, exports, and Manifest ====
validation_results = {}

observed_conditions = set(pipeline_canonical_per_case_metrics_df["stage_condition"])
validation_results["all_five_conditions"] = observed_conditions == set(CANONICAL_CONDITIONS)

stage2_only = pipeline_canonical_per_case_metrics_df.loc[
    pipeline_canonical_per_case_metrics_df["stage_condition"].isin(["P2-Q", "P2-P", "b02", "Full"])
]
observed_families = set(stage2_only["method_family"])
validation_results["all_reranker_families"] = observed_families == set(RERANKER_FAMILIES)

shared_rows = pipeline_canonical_per_case_metrics_df.loc[
    pipeline_canonical_per_case_metrics_df["stage_condition"].isin(["P0", "P1-only"])
]
validation_results["shared_baselines_stored_once"] = (
    set(shared_rows["reranker_method"]) == {"none"}
    and set(shared_rows["method_family"]) == {"shared_retrieval_baseline"}
    and shared_rows["shared_stage1_baseline"].all()
    and not shared_rows["shared_baseline_repeated_for_display"].any()
)

gam_depths = set(
    stage2_only.loc[stage2_only["method_family"].eq("gam"), "candidate_pool_depth"]
    .astype(int)
    .unique()
)
validation_results["gam_fixed_prefix_depths_complete"] = (
    gam_depths == set(CANDIDATE_POOL_DEPTHS)
)
validation_results["all_source_contract_gates_success"] = bool(
    pipeline_source_contract_gate_df["gate_status"].eq("SUCCESS").all()
)
validation_results["pool_depth_and_metric_cutoff_separate"] = (
    "candidate_pool_depth" in pipeline_canonical_per_case_metrics_df.columns
    and "metric_cutoff" in pipeline_canonical_per_case_metrics_df.columns
)

rank_values = pd.to_numeric(canonical_rank_rows_df["target_rank"], errors="coerce")
candidate_counts = pd.to_numeric(canonical_rank_rows_df["candidate_count"], errors="raise")
candidate_depths = pd.to_numeric(canonical_rank_rows_df["candidate_pool_depth"], errors="raise")
validation_results["candidate_counts_within_pool"] = bool(candidate_counts.le(candidate_depths).all())
validation_results["target_rank_one_based"] = bool(rank_values.dropna().ge(1).all())
validation_results["target_rank_within_candidate_count"] = bool(
    rank_values.dropna().le(candidate_counts.loc[rank_values.notna()]).all()
)

absent_metrics = pipeline_canonical_per_case_metrics_df.loc[
    ~pipeline_canonical_per_case_metrics_df["target_exposed"].astype(bool),
    "metric_value",
]
validation_results["target_absent_metrics_zero"] = bool(
    np.isclose(absent_metrics.to_numpy(dtype=float), 0.0, rtol=0.0, atol=0.0).all()
)
validation_results["no_duplicated_canonical_key"] = bool(
    not pipeline_canonical_per_case_metrics_df.duplicated(canonical_key).any()
)
validation_results["no_model_selection"] = True
validation_results["all_source_metrics_validated"] = True
validation_results["source_metric_disagreements_recorded"] = True
validation_results["canonical_metrics_reconstructed_from_target_rank"] = True
validation_results["upstream_readiness_passed"] = bool(
    pipeline_upstream_readiness_df["gate_status"].eq("SUCCESS").all()
)
validation_results["full_transfer_readiness"] = bool(
    pipeline_upstream_readiness_df.loc[
        pipeline_upstream_readiness_df["stage_condition"].eq("Full"),
        [
            "full_validation_passed", "fallback_identity_passed",
            "zero_training_transfer_verified",
            "model_or_checkpoint_sha_consistent",
            "case_count_consistent", "candidate_count_consistent",
        ],
    ].apply(lambda column: boolean_series(column).fillna(False).all()).all()
)
_expected_primary_pair_count = len(
    globals().get(
        "PRIMARY_EXPECTED_METHOD_CONDITION_PAIRS",
        {
            (family, condition)
            for family in RERANKER_FAMILIES
            for condition in ["P0", "P1-only"]
        }.union(
            {
                (source_spec["method_family"], source_spec["stage_condition"])
                for source_spec in SOURCE_RUNS
            }
        ),
    )
)
validation_results["primary_ndcg5_depth1000_grid_complete"] = bool(
    len(pipeline_canonical_primary_ndcg5_depth1000_df)
    == (
        pipeline_canonical_primary_ndcg5_depth1000_df["case_id"].nunique()
        * _expected_primary_pair_count
    )
)
validation_results["canonical_source_hashes_complete"] = bool(
    pipeline_source_hashes_df[
        ["run_manifest_sha256", "per_query_sha256", "rank_artifact_sha256"]
    ].astype(str).ne("").all().all()
)

validation_pool_depth = int(max(CANDIDATE_POOL_DEPTHS))
p0_case_meta_for_validation_df = canonical_rank_rows_df.loc[
    canonical_rank_rows_df["stage_condition"].eq("P0")
    & pd.to_numeric(canonical_rank_rows_df["candidate_pool_depth"], errors="raise").eq(validation_pool_depth)
].drop_duplicates("case_id")
p1_case_meta_for_validation_df = canonical_rank_rows_df.loc[
    canonical_rank_rows_df["stage_condition"].eq("P1-only")
    & pd.to_numeric(canonical_rank_rows_df["candidate_pool_depth"], errors="raise").eq(validation_pool_depth)
].drop_duplicates("case_id")
p0_cases = set(p0_case_meta_for_validation_df["case_id"].astype(str))
p1_cases = set(p1_case_meta_for_validation_df["case_id"].astype(str))
paired_universe_failures = []
for source_spec in SOURCE_RUNS:
    family_condition_rows = canonical_rank_rows_df.loc[
        canonical_rank_rows_df["stage_condition"].eq(source_spec["stage_condition"])
        & canonical_rank_rows_df["method_family"].eq(source_spec["method_family"])
    ]
    expected_cases = p1_cases if source_spec["stage_condition"] in {"Full", "b02"} else p0_cases
    for depth, depth_rows in family_condition_rows.groupby("candidate_pool_depth"):
        observed_cases = set(depth_rows["case_id"].astype(str))
        if observed_cases != expected_cases:
            paired_universe_failures.append({
                "source_notebook": source_spec["source_notebook"],
                "condition": source_spec["stage_condition"],
                "reranker": source_spec["reranker_method"],
                "candidate_pool_depth": int(depth),
                "missing_cases": len(expected_cases.difference(observed_cases)),
                "extra_cases": len(observed_cases.difference(expected_cases)),
            })
validation_results["paired_case_universes_match"] = not paired_universe_failures

expected_fallback_cases = int(stage1_manifest["fallback_case_count"])
if p1_fallback_column_available:
    observed_fallback_cases = int(
        p1_case_meta_for_validation_df.loc[
            boolean_series(p1_case_meta_for_validation_df["profile_fallback_flag"]).fillna(False)
        ]["case_id"].nunique()
    )
    validation_results["qchs_fallback_cases_preserved"] = (
        observed_fallback_cases == expected_fallback_cases
    )
else:
    observed_fallback_cases = None
    validation_results["qchs_fallback_cases_preserved"] = expected_fallback_cases == 0

availability_counts = (
    pipeline_method_depth_availability_df["availability_status"].value_counts().to_dict()
)
validation_results["no_missing_expected_cells"] = (
    int(availability_counts.get("missing_unexpected", 0)) == 0
)

failed_validations = [name for name, passed in validation_results.items() if not bool(passed)]
if failed_validations:
    raise RuntimeError(
        "Canonical pipeline validation failed: "
        + json.dumps({
            "failed_checks": failed_validations,
            "paired_universe_failures": paired_universe_failures[:10],
        }, ensure_ascii=False)
    )

if source_metric_disagreement_df.empty:
    source_metric_disagreement_summary_df = pd.DataFrame(
        columns=[
            "source_notebook", "condition", "reranker",
            "candidate_pool_depth", "source_metric_name",
            "disagreement_count", "headline_depth_1000",
        ]
    )
else:
    source_metric_disagreement_summary_df = (
        source_metric_disagreement_df
        .groupby(
            [
                "source_notebook", "condition", "reranker",
                "candidate_pool_depth", "source_metric_name",
            ],
            dropna=False,
        )
        .size()
        .rename("disagreement_count")
        .reset_index()
        .sort_values(
            ["candidate_pool_depth", "condition", "reranker", "source_metric_name"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )
    source_metric_disagreement_summary_df["headline_depth_1000"] = (
        pd.to_numeric(
            source_metric_disagreement_summary_df["candidate_pool_depth"],
            errors="raise",
        ).eq(REPORT_POOL_DEPTH)
    )

STAGED_OUTPUT_FILES = {
    name: RUN_STAGING_DIR / path.name
    for name, path in OUTPUT_FILES.items()
}

pipeline_canonical_per_case_metrics_df.to_parquet(
    STAGED_OUTPUT_FILES["canonical_raw"], index=False
)
pipeline_canonical_per_case_metrics_df.head(2000).to_csv(
    STAGED_OUTPUT_FILES["canonical_raw_preview"],
    index=False,
    encoding="utf-8-sig",
)
# --- display LABEL_MAP (SSoT §4): machine stage_condition -> display name ---
LABEL_MAP = {"P0": "S1-Q", "P1-only": "S1-P", "P2-Q": "Base", "P2-P": "RankP", "b02": "RetrP", "Full": "Full"}
pipeline_canonical_primary_ndcg5_depth1000_df["condition_display"] = pipeline_canonical_primary_ndcg5_depth1000_df["stage_condition"].map(LABEL_MAP)
pipeline_canonical_primary_ndcg5_depth1000_df.to_parquet(
    STAGED_OUTPUT_FILES["canonical_primary"], index=False
)
pipeline_canonical_primary_ndcg5_depth1000_df.head(2000).to_csv(
    STAGED_OUTPUT_FILES["canonical_primary_preview"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_results_by_regime_pool_depth_df.to_csv(
    STAGED_OUTPUT_FILES["by_regime_pool_depth"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_results_by_regime_df.to_csv(
    STAGED_OUTPUT_FILES["by_regime"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_results_by_pool_depth_df.to_csv(
    STAGED_OUTPUT_FILES["by_pool_depth"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_results_overall_df.to_csv(
    STAGED_OUTPUT_FILES["overall"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_five_condition_comparison_df.to_csv(
    STAGED_OUTPUT_FILES["five_condition_comparison"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_method_depth_availability_df.to_csv(
    STAGED_OUTPUT_FILES["method_depth_availability"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_source_inventory_df.to_csv(
    STAGED_OUTPUT_FILES["source_inventory"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_source_contract_gate_df.to_csv(
    STAGED_OUTPUT_FILES["source_contract_gate"],
    index=False,
    encoding="utf-8-sig",
)
source_metric_disagreement_df.to_csv(
    STAGED_OUTPUT_FILES["source_metric_disagreements"],
    index=False,
    encoding="utf-8-sig",
)
source_metric_disagreement_summary_df.to_csv(
    STAGED_OUTPUT_FILES["source_metric_disagreement_summary"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_upstream_readiness_df.to_csv(
    STAGED_OUTPUT_FILES["upstream_readiness"],
    index=False,
    encoding="utf-8-sig",
)
pipeline_source_hashes_df.to_csv(
    STAGED_OUTPUT_FILES["source_hashes"],
    index=False,
    encoding="utf-8-sig",
)

canonical_raw_sha256 = file_sha256(STAGED_OUTPUT_FILES["canonical_raw"])
canonical_primary_sha256 = file_sha256(STAGED_OUTPUT_FILES["canonical_primary"])


manifest = {
    "run_status": "SUCCESS",
    "contract_version": "pipeline_aggregate_v4_final_interface_reconciled",
    "run_id": RUN_ID,
    "publication_state": "published_after_all_blocking_qc",
    "interpretation_artifacts_required_for_canonical_aggregation": False,
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_label": CATEGORY_LABEL,
    "project_root": str(PROJECT_ROOT),
    "analysis_output_dir": str(OUT_DIR),
    "canonical_conditions": CANONICAL_CONDITIONS,
    "headline_authority": {
        "metric_name": PRIMARY_METRIC_NAME,
        "metric_cutoff": PRIMARY_METRIC_CUTOFF,
        "candidate_pool_depth": REPORT_POOL_DEPTH,
        "unconditional": True,
        "complete_method_grid_path": str(OUTPUT_FILES["canonical_primary"]),
        "complete_method_grid_sha256": canonical_primary_sha256,
    },
    "canonical_raw_sha256": canonical_raw_sha256,
    "canonical_primary_sha256": canonical_primary_sha256,
    "canonical_source_hashes_path": str(OUTPUT_FILES["source_hashes"]),
    "upstream_readiness_path": str(OUTPUT_FILES["upstream_readiness"]),
    "full_transfer_readiness": bool(validation_results["full_transfer_readiness"]),
    "ready_for_downstream": bool(
        validation_results["upstream_readiness_passed"]
        and validation_results["full_transfer_readiness"]
        and validation_results["primary_ndcg5_depth1000_grid_complete"]
        and validation_results["canonical_source_hashes_complete"]
    ),
    "reranker_families": RERANKER_FAMILIES,
    "pool_depth_contract": {
        "candidate_pool_depths": CANDIDATE_POOL_DEPTHS,
        "metric_cutoffs": METRIC_CUTOFFS,
        "dimensions_are_distinct": True,
    },
    "metric_formulas": METRIC_FORMULAS,
    "gam_depth_policy": {
        "fit_depth": 1000,
        "available_candidate_pool_depths": GAM_AVAILABLE_CANDIDATE_POOL_DEPTHS,
        "evaluation_policy": "fit_once_at_1000_and_evaluate_deterministic_fixed_prefixes",
    },
    "shared_baseline_policy": {
        "canonical_reranker_method": "none",
        "canonical_method_family": "shared_retrieval_baseline",
        "stored_once_in_raw_data": True,
        "repeated_only_in_five_condition_display": True,
        "display_rows_excluded_from_inferential_data": True,
    },
    "notebook09_manifest_path": str(STAGE1_MANIFEST_PATH),
    "stage1_winner_lineage": {
        "notebook07_winner_manifest_path": str(notebook07_winner_manifest_path),
        "notebook08_winner_manifest_path": str(notebook08_winner_manifest_path),
        "query_only_method_key": stage1_manifest.get("baseline_retrieval_winner_method_key"),
        "personalized_method_slug": stage1_manifest.get("selected_personalized_method_slug"),
        "lineage_validated": True,
    },
    "output_paths": {name: str(path) for name, path in OUTPUT_FILES.items()},
    "raw_row_count": int(len(pipeline_canonical_per_case_metrics_df)),
    "unique_case_count": int(pipeline_canonical_per_case_metrics_df["case_id"].nunique()),
    "regime_counts": json.loads(regime_counts_json(pipeline_canonical_per_case_metrics_df)),
    "source_inventory": pipeline_source_inventory_df.to_dict(orient="records"),
    "source_contract_gate": pipeline_source_contract_gate_df.to_dict(orient="records"),
    "source_contract_gate_path": str(OUTPUT_FILES["source_contract_gate"]),
    "source_metric_disagreement_path": str(
        OUTPUT_FILES["source_metric_disagreements"]
    ),
    "source_metric_disagreement_count": int(
        len(source_metric_disagreement_df)
    ),
    "source_metric_disagreements": {
        "path": str(OUTPUT_FILES["source_metric_disagreements"]),
        "summary_path": str(OUTPUT_FILES["source_metric_disagreement_summary"]),
        "row_count": int(len(source_metric_disagreement_df)),
        "headline_depth_1000_row_count": int(
            pd.to_numeric(
                source_metric_disagreement_df.get(
                    "candidate_pool_depth",
                    pd.Series(dtype=float),
                ),
                errors="coerce",
            ).eq(REPORT_POOL_DEPTH).sum()
        ),
        "policy": "record_only_canonical_metrics_reconstructed_from_target_rank",
    },
    "canonical_source_hashes": pipeline_source_hashes_df.to_dict(orient="records"),
    "upstream_readiness": pipeline_upstream_readiness_df.to_dict(orient="records"),
    "missing_expected_cells": pipeline_method_depth_availability_df.loc[
        pipeline_method_depth_availability_df["availability_status"].eq("missing_unexpected")
    ].to_dict(orient="records"),
    "structural_unavailable_cells": {
        "availability_status": "not_run_by_design",
        "cell_count": int(availability_counts.get("not_run_by_design", 0)),
        "design_excluded_method_condition_pairs": [
            {"reranker_family": _excluded_family, "stage_condition": _excluded_condition}
            for _excluded_family, _excluded_condition in DESIGN_EXCLUDED_METHOD_CONDITION_PAIRS
        ],
        "rule": (
            "Every (family, condition) cell instantiated by the 2x2 stage allocation "
            "reports the canonical fixed-prefix depth grid. Cells absent from SOURCE_RUNS "
            "are absent by design, not by fault, and are excluded from "
            "no_missing_expected_cells."
        ),
    },
    "invalid_cutoff_cells": int(availability_counts.get("invalid_cutoff_for_pool", 0)),
    "qchs_fallback_preservation": {
        "notebook09_flag_available": bool(p1_fallback_column_available),
        "qchs_profile_available_column_present": bool(p1_qchs_column_available),
        "expected_fallback_case_count": expected_fallback_cases,
        "observed_fallback_case_count": observed_fallback_cases,
        "preserved": bool(validation_results["qchs_fallback_cases_preserved"]),
    },
    "validation_results": validation_results,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
write_manifest(STAGED_OUTPUT_FILES["manifest"], manifest)

_staged_manifest = json.loads(
    STAGED_OUTPUT_FILES["manifest"].read_text(encoding="utf-8")
)
if _staged_manifest.get("run_status") != "SUCCESS":
    raise RuntimeError("Notebook 14 staged SUCCESS manifest gate failed.")
if _staged_manifest.get("ready_for_downstream") is not True:
    raise RuntimeError("Notebook 14 staged output is not downstream-ready.")
if _staged_manifest.get("canonical_raw_sha256") != file_sha256(
    STAGED_OUTPUT_FILES["canonical_raw"]
):
    raise RuntimeError("Notebook 14 staged canonical raw hash changed before promotion.")
if _staged_manifest.get("canonical_primary_sha256") != file_sha256(
    STAGED_OUTPUT_FILES["canonical_primary"]
):
    raise RuntimeError("Notebook 14 staged primary-grid hash changed before promotion.")

# Promote all data/QC outputs first and the SUCCESS manifest last.
for _name, _staged_path in STAGED_OUTPUT_FILES.items():
    if _name == "manifest":
        continue
    if not Path(_staged_path).exists():
        raise FileNotFoundError(f"Missing staged Notebook 14 output: {_staged_path}")
    Path(OUTPUT_FILES[_name]).parent.mkdir(parents=True, exist_ok=True)
    os.replace(_staged_path, OUTPUT_FILES[_name])

os.replace(STAGED_OUTPUT_FILES["manifest"], OUTPUT_FILES["manifest"])
if RUN_STAGING_DIR.exists():
    shutil.rmtree(RUN_STAGING_DIR)

reloaded_manifest = json.loads(OUTPUT_FILES["manifest"].read_text(encoding="utf-8"))
if reloaded_manifest.get("run_status") != "SUCCESS":
    raise RuntimeError("Notebook 14 SUCCESS manifest gate failed.")
if reloaded_manifest.get("ready_for_downstream") is not True:
    raise RuntimeError("Notebook 14 is not ready for Notebooks 15 and 16.")
if reloaded_manifest.get("canonical_raw_sha256") != file_sha256(
    OUTPUT_FILES["canonical_raw"]
):
    raise RuntimeError("Notebook 14 canonical raw hash changed after sealing.")
if reloaded_manifest.get("canonical_primary_sha256") != file_sha256(
    OUTPUT_FILES["canonical_primary"]
):
    raise RuntimeError("Notebook 14 primary-grid hash changed after sealing.")

print("Canonical pipeline aggregation saved to:", OUT_DIR)
print("Raw rows:", len(pipeline_canonical_per_case_metrics_df))
print("Unique cases:", pipeline_canonical_per_case_metrics_df["case_id"].nunique())
print("Validation:", validation_results)


Canonical pipeline aggregation saved to: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/pipeline_aggregate
Raw rows: 3011040
Unique cases: 1968
Validation: {'all_five_conditions': True, 'all_reranker_families': True, 'shared_baselines_stored_once': True, 'gam_fixed_prefix_depths_complete': True, 'all_source_contract_gates_success': True, 'pool_depth_and_metric_cutoff_separate': True, 'candidate_counts_within_pool': True, 'target_rank_one_based': True, 'target_rank_within_candidate_count': True, 'target_absent_metrics_zero': True, 'no_duplicated_canonical_key': True, 'no_model_selection': True, 'all_source_metrics_validated': True, 'source_metric_disagreements_recorded': True, 'canonical_metrics_reconstructed_from_target_rank': True, 'upstream_readiness_passed': True, 'full_transfer_readiness': True, 'primary_ndcg5_depth1000_grid_complete': True, 'canonical_source_hashes_complete': True, 'paired_case_universes_match': True, 'qchs_fallback_cases_preserved': Tr

In [13]:
# ==== Descriptive Secondary-Metric Reference Export ====
_w23 = pipeline_canonical_per_case_metrics_df.copy()
_w23["metric_value"] = pd.to_numeric(_w23["metric_value"], errors="raise")
_w23_grp = (
    _w23.groupby(
        ["stage_condition", "method_family", "candidate_pool_depth", "metric_name", "metric_cutoff"],
        observed=True, dropna=False,
    )["metric_value"].agg(mean_metric_value="mean", n_cases="size").reset_index()
)
_w23_grp.insert(0, "category_id", CATEGORY_ID)
_w23_grp["metric"] = _w23_grp["metric_name"].astype(str) + "@" + _w23_grp["metric_cutoff"].astype(int).astype(str)

# Chapter-6 reference set: Stage-1 exposure @1000; Stage-2 headline-depth-1000 secondary metrics.
_STAGE1 = {"P0", "P1-only"}
_STAGE2 = {"P2-Q", "P2-P", "b02", "Full"}  # PATCH 2026-07-26: b02 (RetrP) was silently excluded from the Ch.6 reference table
_S1_SET = {("HitRate", 1000), ("NDCG", 1000), ("MRR", 1000)}
_S2_SET = {("NDCG", 1), ("NDCG", 5), ("HitRate", 1), ("HitRate", 5), ("MRR", 5)}

def _w23_in_ref(row):
    sc = str(row["stage_condition"]); nm = str(row["metric_name"])
    ct = int(row["metric_cutoff"]); dp = int(row["candidate_pool_depth"])
    if sc in _STAGE1:
        return dp == 1000 and (nm, ct) in _S1_SET
    if sc in _STAGE2:
        return dp == 1000 and (nm, ct) in _S2_SET
    return False

_w23_grp["in_ch6_reference_table"] = _w23_grp.apply(_w23_in_ref, axis=1)
_w23_grp["condition_display"] = _w23_grp["stage_condition"].map(
    {"P0": "S1-Q", "P1-only": "S1-P", "P2-Q": "Base", "P2-P": "RankP", "b02": "RetrP", "Full": "Full"}
)  # PATCH 2026-07-26: SSoT §4 display layer
_w23_grp["inference_role"] = "descriptive_exploratory"
_w23_grp["confirmatory_metric"] = (
    _w23_grp["metric_name"].eq("NDCG")
    & _w23_grp["metric_cutoff"].eq(5)
    & pd.to_numeric(_w23_grp["candidate_pool_depth"], errors="raise").eq(1000)
)
overall_results_table_df = _w23_grp.sort_values(
    ["stage_condition", "method_family", "candidate_pool_depth", "metric_name", "metric_cutoff"],
    kind="mergesort",
).reset_index(drop=True)
_w23_path = OUT_DIR / "overall_results_table.csv"
overall_results_table_df.to_csv(_w23_path, index=False, encoding="utf-8-sig")
print("W2-3 overall_results_table:", str(_w23_path), "| rows:", len(overall_results_table_df),
      "| ch6-reference rows:", int(overall_results_table_df["in_ch6_reference_table"].sum()))
try:
    display(overall_results_table_df.loc[overall_results_table_df["in_ch6_reference_table"]])
except NameError:
    print(overall_results_table_df.loc[overall_results_table_df["in_ch6_reference_table"]].to_string(index=False))


W2-3 overall_results_table: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/pipeline_aggregate/overall_results_table.csv | rows: 1530 | ch6-reference rows: 81


,category_id,stage_condition,method_family,candidate_pool_depth,metric_name,metric_cutoff,mean_metric_value,n_cases,metric,in_ch6_reference_table,condition_display,inference_role,confirmatory_metric
66,herbal,Full,gam,1000,HitRate,1,0.014228,1968,HitRate@1,True,Full,descriptive_exploratory,False
67,herbal,Full,gam,1000,HitRate,5,0.052337,1968,HitRate@5,True,Full,descriptive_exploratory,False
75,herbal,Full,gam,1000,MRR,5,0.027354,1968,MRR@5,True,Full,descriptive_exploratory,False
82,herbal,Full,gam,1000,NDCG,1,0.014228,1968,NDCG@1,True,Full,descriptive_exploratory,False
83,herbal,Full,gam,1000,NDCG,5,0.033524,1968,NDCG@5,True,Full,descriptive_exploratory,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,herbal,b02,transformer,1000,HitRate,1,0.027439,1968,HitRate@1,True,RetrP,descriptive_exploratory,False
1507,herbal,b02,transformer,1000,HitRate,5,0.086382,1968,HitRate@5,True,RetrP,descriptive_exploratory,False
1515,herbal,b02,transformer,1000,MRR,5,0.047815,1968,MRR@5,True,RetrP,descriptive_exploratory,False
1522,herbal,b02,transformer,1000,NDCG,1,0.027439,1968,NDCG@1,True,RetrP,descriptive_exploratory,False
